# 🦠 EpiAgent: Secure Multi-Agent Epidemic Surveillance & Predictive Modeling System**Google/Kaggle 5-Day AI Agents Intensive Course — "Agents for Good" (Healthcare)**---## Executive SummaryEpiAgent is an autonomous public health intelligence pipeline that:1. **Streams** surveillance data from synthetic SEIR models or CDC APIs2. **Secures** data with HIPAA-compliant PII detection (18 identifier types)3. **Validates** data quality with 8-point epidemiological plausibility checks4. **Analyzes** epidemics using SEIR models, Bayesian Rt estimation, and BOCPD changepoint detection5. **Forecasts** case counts using XGBoost ensemble with SHAP explainability6. **Reports** executive situation reports with alert level classification### Key Innovation: "LLM Decides, Math Computes, LLM Interprets"Every numerical result comes from a deterministic `FunctionTool` — never from LLM generation. The agents decide *when* to call tools and *how to interpret* results, but the math is always exact and reproducible.### Architecture```Data Agent → Security Agent → Validator Agent → Analysis Agent → ML Agent → SitRep Agent                                                                                    ↓                                                                        Interactive Dashboard                                                                        Executive SitRep```

In [ ]:
# Install required packages!pip install -q numpy scipy pandas pydantic xgboost scikit-learn shap plotly requests google-adk "mcp[cli]"import warningswarnings.filterwarnings('ignore')import jsonimport loggingimport hashlibimport reimport timeimport uuidfrom dataclasses import dataclass, fieldfrom datetime import date, datetime, timedeltafrom enum import Enumfrom typing import Any, Literalimport numpy as npimport pandas as pdimport plotly.graph_objects as gofrom plotly.subplots import make_subplotsfrom scipy import statsfrom scipy.integrate import solve_ivpfrom scipy.stats import gamma as gamma_distprint("✅ All packages imported successfully")print(f"NumPy: {np.__version__}, Pandas: {pd.__version__}, SciPy: {stats.scipy.__version__}")

## 📐 Statistical Methods### 1. SEIR Compartmental ModelThe SEIR model divides the population into four compartments:$$\frac{dS}{dt} = -\beta \frac{SI}{N}, \quad \frac{dE}{dt} = \beta \frac{SI}{N} - \sigma E, \quad \frac{dI}{dt} = \sigma E - \gamma I, \quad \frac{dR}{dt} = \gamma I$$Where: $\beta = R_0 \cdot \gamma$, $\sigma = 1/\text{latent period}$, $\gamma = 1/\text{infectious period}$.### 2. Bayesian Rt Estimation (Cori et al., 2013)The instantaneous reproduction number using Gamma conjugate prior:$$R_t \mid \text{data} \sim \text{Gamma}\left(a + \sum_{s \in \tau} I_s, \; \left(\frac{1}{b} + \sum_{s \in \tau} \Lambda_s\right)^{-1}\right)$$### 3. Wilson Score Confidence IntervalFor CFR and attack rate (superior to Wald interval):$$\text{CI} = \frac{\hat{p} + \frac{z^2}{2n} \pm z\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}$$### 4. BOCPD Changepoint Detection (Adams & MacKay, 2007)Bayesian Online Changepoint Detection with Student-t predictive distribution.### 5. XGBoost + SHAP Ensemble ForecastingFeature-engineered gradient boosted trees with SHAP TreeExplainer for post-hoc explainability.

## 🔧 Core Engine CodeThe following cells contain the deterministic computation engines.

### SEIR Model`epiagent/engines/seir_model.py`

In [ ]:
"""SEIR Compartmental Model for Epidemic Simulation.Implements the Susceptible-Exposed-Infectious-Recovered model usingscipy.integrate.solve_ivp with RK45 method.Differential Equations:    dS/dt = -β·S·I/N    dE/dt =  β·S·I/N - σ·E    dI/dt =  σ·E - γ·I    dR/dt =  γ·IWhere:    β = transmission rate = R0 · γ    σ = 1/latent_period (rate of progression from E→I)    γ = 1/infectious_period (recovery rate)    R0 = basic reproduction number = β/γReferences:    Kermack WO, McKendrick AG. (1927) "A Contribution to the Mathematical    Theory of Epidemics." Proceedings of the Royal Society A, 115(772):700-721."""from __future__ import annotationsimport loggingfrom dataclasses import dataclass, fieldimport numpy as npfrom scipy.integrate import solve_ivpfrom scipy.optimize import minimizelogger = logging.getLogger(__name__)@dataclassclass SEIRParameters:    """Parameters for the SEIR compartmental model.    Attributes:        beta: Transmission rate. Related to R0 by β = R0 · γ.        sigma: Rate of progression from Exposed to Infectious (1/latent_period).        gamma: Recovery rate (1/infectious_period).        N: Total population (constant).        S0: Initial number of susceptible individuals.        E0: Initial number of exposed individuals.        I0: Initial number of infectious individuals.        R0_init: Initial number of recovered individuals.    """    beta: float    sigma: float    gamma: float    N: int    S0: float = 0.0    E0: float = 0.0    I0: float = 10.0    R0_init: float = 0.0    def __post_init__(self):        if self.S0 == 0.0:            self.S0 = self.N - self.E0 - self.I0 - self.R0_init    @property    def R0(self) -> float:        """Basic reproduction number."""        if self.gamma == 0:            return float("inf")        return self.beta / self.gamma    @property    def latent_period(self) -> float:        """Average latent period in days."""        if self.sigma == 0:            return float("inf")        return 1.0 / self.sigma    @property    def infectious_period(self) -> float:        """Average infectious period in days."""        if self.gamma == 0:            return float("inf")        return 1.0 / self.gamma    @property    def serial_interval(self) -> float:        """Serial interval = latent_period + infectious_period."""        return self.latent_period + self.infectious_period    @classmethod    def from_epi_params(        cls,        R0: float,        latent_period: float,        infectious_period: float,        population: int,        initial_infected: int = 10,        initial_exposed: int = 0,    ) -> SEIRParameters:        """Create parameters from intuitive epidemiological values.        Args:            R0: Basic reproduction number.            latent_period: Average latent period in days.            infectious_period: Average infectious period in days.            population: Total population.            initial_infected: Initial infectious count.            initial_exposed: Initial exposed count.        Returns:            SEIRParameters instance.        """        gamma = 1.0 / infectious_period        sigma = 1.0 / latent_period        beta = R0 * gamma        return cls(            beta=beta,            sigma=sigma,            gamma=gamma,            N=population,            E0=float(initial_exposed),            I0=float(initial_infected),            R0_init=0.0,        )@dataclassclass SEIRResult:    """Results from SEIR model simulation.    Attributes:        t: Time points (days).        S: Susceptible compartment over time.        E: Exposed compartment over time.        I: Infectious compartment over time.        R: Recovered compartment over time.        daily_incidence: Daily new infections (flow from S→E).        R0: Basic reproduction number used.        peak_day: Day of peak infections.        peak_cases: Maximum number of infectious individuals.    """    t: np.ndarray    S: np.ndarray    E: np.ndarray    I: np.ndarray    R: np.ndarray    daily_incidence: np.ndarray    R0: float    peak_day: int    peak_cases: float    def summary(self) -> dict:        """Return concise summary dict for agent state."""        return {            "R0": round(self.R0, 2),            "peak_day": int(self.peak_day),            "peak_cases": int(round(self.peak_cases)),            "total_infected": int(round(self.R[-1])) if len(self.R) > 0 else 0,            "attack_rate": round(float(self.R[-1] / (self.S[0] + self.E[0] + self.I[0] + self.R[0])), 4)            if len(self.R) > 0            else 0.0,            "duration_days": int(self.t[-1]) if len(self.t) > 0 else 0,        }def _seir_odes(t: float, y: list[float], N: int, beta: float, sigma: float, gamma: float):    """SEIR system of ordinary differential equations.    Args:        t: Current time (not used explicitly, required by solve_ivp).        y: State vector [S, E, I, R].        N: Total population.        beta: Transmission rate.        sigma: Incubation rate (1/latent_period).        gamma: Recovery rate (1/infectious_period).    Returns:        List of derivatives [dS/dt, dE/dt, dI/dt, dR/dt].    """    S, E, I, R = y    # Force of infection    force_of_infection = beta * S * I / N    dS = -force_of_infection    dE = force_of_infection - sigma * E    dI = sigma * E - gamma * I    dR = gamma * I    return [dS, dE, dI, dR]def run_seir(params: SEIRParameters, t_max: int = 365) -> SEIRResult:    """Run the SEIR model forward simulation.    Args:        params: SEIRParameters defining the model.        t_max: Maximum simulation time in days.    Returns:        SEIRResult with full compartment trajectories.    Raises:        ValueError: If parameters are invalid (negative rates, etc.).    """    # Validate parameters    if params.beta < 0 or params.sigma < 0 or params.gamma < 0:        raise ValueError(            f"Rates must be non-negative: beta={params.beta}, "            f"sigma={params.sigma}, gamma={params.gamma}"        )    if params.N <= 0:        raise ValueError(f"Population must be positive: N={params.N}")    # Handle edge case: no initial infections → no epidemic    if params.I0 == 0 and params.E0 == 0:        t = np.arange(t_max + 1, dtype=float)        n = len(t)        return SEIRResult(            t=t,            S=np.full(n, params.S0),            E=np.zeros(n),            I=np.zeros(n),            R=np.full(n, params.R0_init),            daily_incidence=np.zeros(n),            R0=params.R0,            peak_day=0,            peak_cases=0.0,        )    # Initial conditions    y0 = [params.S0, params.E0, params.I0, params.R0_init]    # Time span    t_span = (0, t_max)    t_eval = np.arange(t_max + 1, dtype=float)    # Solve ODE system    sol = solve_ivp(        fun=_seir_odes,        t_span=t_span,        y0=y0,        args=(params.N, params.beta, params.sigma, params.gamma),        t_eval=t_eval,        method="RK45",        rtol=1e-8,        atol=1e-8,    )    if not sol.success:        raise RuntimeError(f"ODE solver failed: {sol.message}")    S, E, I, R = sol.y    # Conservation law check: S + E + I + R = N    total = S + E + I + R    max_deviation = np.max(np.abs(total - params.N))    if max_deviation > 1e-3:        logger.warning(            "Conservation law violation: max deviation = %.6f (threshold=1e-3)",            max_deviation,        )    # Compute daily incidence (new infections per day = flow from S to E)    daily_incidence = compute_daily_incidence(S, params.beta, I, params.N)    # Find peak    peak_idx = int(np.argmax(I))    result = SEIRResult(        t=sol.t,        S=S,        E=E,        I=I,        R=R,        daily_incidence=daily_incidence,        R0=params.R0,        peak_day=peak_idx,        peak_cases=float(I[peak_idx]),    )    logger.info(        "SEIR simulation complete: R0=%.2f, peak at day %d with %d cases",        params.R0, peak_idx, int(I[peak_idx]),    )    return resultdef compute_daily_incidence(    S: np.ndarray, beta: float, I: np.ndarray, N: int) -> np.ndarray:    """Compute daily new infections from SEIR compartments.    Daily incidence = β·S·I/N (the flow from S to E per unit time).    Args:        S: Susceptible compartment array.        beta: Transmission rate.        I: Infectious compartment array.        N: Total population.    Returns:        Array of daily incidence values.    """    if N == 0:        return np.zeros_like(S)    incidence = beta * S * I / N    return np.maximum(incidence, 0.0)def fit_seir(    observed_cases: np.ndarray,    population: int,    initial_guess: dict | None = None,    method: str = "Nelder-Mead",) -> tuple[SEIRParameters, SEIRResult, dict]:    """Fit SEIR model parameters to observed daily case counts.    Uses scipy.optimize.minimize to find β, σ, γ that minimize the    RMSE between model-predicted daily incidence and observed cases.    Args:        observed_cases: Array of daily observed case counts.        population: Total population.        initial_guess: Optional dict with keys 'R0', 'latent_period',            'infectious_period'. Defaults to COVID-like parameters.        method: Optimization method (default: Nelder-Mead).    Returns:        Tuple of (fitted_params, fitted_result, fit_metrics).        fit_metrics contains 'rmse', 'r_squared', 'n_iterations'.    """    observed = np.asarray(observed_cases, dtype=float)    T = len(observed)    if T < 7:        raise ValueError(f"Need at least 7 data points for fitting, got {T}")    # Default initial guess: COVID-like parameters    if initial_guess is None:        initial_guess = {            "R0": 2.5,            "latent_period": 5.2,            "infectious_period": 2.9,        }    # Estimate initial infected from first non-zero observation    first_nonzero = np.argmax(observed > 0)    I0 = max(observed[first_nonzero], 1.0) if first_nonzero < T else 1.0    def objective(x):        """Objective function: RMSE between model and observed."""        R0_est, lat_est, inf_est = x        # Parameter bounds enforcement        if R0_est <= 0 or lat_est <= 0 or inf_est <= 0:            return 1e10        try:            params = SEIRParameters.from_epi_params(                R0=R0_est,                latent_period=lat_est,                infectious_period=inf_est,                population=population,                initial_infected=int(I0),            )            result = run_seir(params, t_max=T - 1)            # Compare daily incidence            model_incidence = result.daily_incidence[:T]            if len(model_incidence) < T:                return 1e10            # RMSE            rmse = np.sqrt(np.mean((model_incidence - observed) ** 2))            return rmse        except (ValueError, RuntimeError):            return 1e10    # Initial parameter vector: [R0, latent_period, infectious_period]    x0 = [        initial_guess["R0"],        initial_guess["latent_period"],        initial_guess["infectious_period"],    ]    # Optimize    opt_result = minimize(        objective,        x0=x0,        method=method,        options={"maxiter": 2000, "xatol": 1e-6, "fatol": 1e-6},    )    # Extract fitted parameters    R0_fit, lat_fit, inf_fit = opt_result.x    # Clamp to reasonable ranges    R0_fit = max(0.1, min(R0_fit, 30.0))    lat_fit = max(0.5, min(lat_fit, 30.0))    inf_fit = max(0.5, min(inf_fit, 30.0))    fitted_params = SEIRParameters.from_epi_params(        R0=R0_fit,        latent_period=lat_fit,        infectious_period=inf_fit,        population=population,        initial_infected=int(I0),    )    fitted_result = run_seir(fitted_params, t_max=T - 1)    # Compute fit metrics    model_incidence = fitted_result.daily_incidence[:T]    rmse = float(np.sqrt(np.mean((model_incidence - observed) ** 2)))    # R-squared    ss_res = np.sum((observed - model_incidence) ** 2)    ss_tot = np.sum((observed - np.mean(observed)) ** 2)    r_squared = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else 0.0    fit_metrics = {        "rmse": round(rmse, 4),        "r_squared": round(r_squared, 4),        "n_iterations": int(opt_result.nit),        "converged": bool(opt_result.success),        "fitted_R0": round(R0_fit, 3),        "fitted_latent_period": round(lat_fit, 3),        "fitted_infectious_period": round(inf_fit, 3),    }    logger.info(        "SEIR fitting complete: R0=%.2f, RMSE=%.2f, R²=%.4f, converged=%s",        R0_fit, rmse, r_squared, opt_result.success,    )    return fitted_params, fitted_result, fit_metrics

### Rt Estimation`epiagent/engines/rt_estimation.py`

In [ ]:
"""Bayesian Real-Time Effective Reproduction Number (Rt) Estimation.Implements the Cori et al. (2013) method — the WHO/CDC gold standardfor real-time Rt estimation during epidemics.Reference:    Cori A, Ferguson NM, Fraser C, Cauchemez S. (2013)    "A New Framework and Software to Estimate Time-Varying    Reproduction Numbers During Epidemics."    American Journal of Epidemiology, 178(9):1505-1512.Method:    The instantaneous reproduction number Rt is estimated using a    Bayesian framework with a Gamma conjugate prior:    Likelihood: I_t ~ Poisson(R_t · Λ_t)    Prior: R_t ~ Gamma(a, scale=b)    Posterior: R_t | data ~ Gamma(a_post, scale=b_post)    Where:        a_post = a_prior + Σ I_t  (sum of cases over sliding window)        b_post = 1 / (1/b_prior + Σ Λ_t)  (total infectiousness)        Λ_t = Σ_{s=1}^{T} I_{t-s} · w_s  (total infectiousness at time t)        w_s = serial interval distribution (discretized Gamma PMF)    The 95% credible interval is computed from the Gamma posterior quantiles."""from __future__ import annotationsimport loggingfrom dataclasses import dataclass, fieldimport numpy as npfrom scipy.stats import gamma as gamma_distlogger = logging.getLogger(__name__)@dataclassclass SerialInterval:    """Serial interval distribution for a pathogen.    The serial interval is the time between symptom onset in a primary    case and symptom onset in a secondary case.    Attributes:        mean: Mean serial interval in days.        std: Standard deviation of serial interval in days.    """    mean: float    std: float    def discretize(self, max_days: int = 20) -> np.ndarray:        """Discretize the serial interval into a probability mass function.        Uses a Gamma distribution parameterized by the mean and std,        evaluated at integer day values and normalized to sum to 1.        Args:            max_days: Maximum number of days to include in the PMF.        Returns:            Array of probabilities: w[s] = P(serial interval = s days).            w[0] = 0 by convention (no same-day transmission).        """        if self.mean <= 0 or self.std <= 0:            raise ValueError(                f"Mean and std must be positive: mean={self.mean}, std={self.std}"            )        # Gamma shape and scale from mean and std        # mean = shape * scale, var = shape * scale^2        # → shape = (mean/std)^2, scale = std^2/mean        shape = (self.mean / self.std) ** 2        scale = self.std ** 2 / self.mean        # Evaluate Gamma PDF at integer day values        days = np.arange(0, max_days + 1)        pmf = gamma_dist.pdf(days, a=shape, scale=scale)        # Zero out day 0 (no same-day transmission)        pmf[0] = 0.0        # Normalize to ensure PMF sums to 1        total = pmf.sum()        if total > 0:            pmf /= total        else:            # Fallback: uniform distribution if parameterization fails            pmf[1:] = 1.0 / max_days        return pmf# ---------------------------------------------------------------------------# Preset serial interval distributions# ---------------------------------------------------------------------------SI_INFLUENZA = SerialInterval(mean=2.6, std=1.5)"""Influenza serial interval. Ref: Ferguson et al. (2005)"""SI_COVID = SerialInterval(mean=4.7, std=2.9)"""COVID-19 serial interval. Ref: Nishiura et al. (2020)"""SI_MEASLES = SerialInterval(mean=11.5, std=2.0)"""Measles serial interval. Ref: Fine (2003)"""@dataclassclass RtResult:    """Result of Bayesian Rt estimation.    Attributes:        t: Time indices (aligned with input incidence array).        rt_mean: Posterior mean of Rt at each time step.        rt_lower: Lower bound of 95% credible interval.        rt_upper: Upper bound of 95% credible interval.        epidemic_phase: Classification at each step: 'growing', 'declining', 'stable'.        posterior_shape: Posterior Gamma shape parameter at each step.        posterior_scale: Posterior Gamma scale parameter at each step.    """    t: np.ndarray    rt_mean: np.ndarray    rt_lower: np.ndarray    rt_upper: np.ndarray    epidemic_phase: list[str]    posterior_shape: np.ndarray    posterior_scale: np.ndarray    @property    def current_rt(self) -> float:        """Most recent Rt estimate (last non-NaN value)."""        valid = ~np.isnan(self.rt_mean)        if valid.any():            return float(self.rt_mean[valid][-1])        return float("nan")    @property    def current_phase(self) -> str:        """Most recent epidemic phase classification."""        for phase in reversed(self.epidemic_phase):            if phase in ("growing", "declining", "stable"):                return phase        return "unknown"    def summary(self) -> dict:        """Return concise summary dict for agent state."""        return {            "current_rt": round(self.current_rt, 3),            "current_phase": self.current_phase,            "rt_range": [                round(float(np.nanmin(self.rt_mean)), 3),                round(float(np.nanmax(self.rt_mean)), 3),            ],            "n_valid_estimates": int(np.sum(~np.isnan(self.rt_mean))),        }def estimate_rt(    incidence: np.ndarray,    serial_interval: SerialInterval,    window: int = 7,    prior_shape: float = 1.0,    prior_scale: float = 5.0,    confidence: float = 0.95,) -> RtResult:    """Estimate time-varying Rt using the Cori et al. (2013) method.    Args:        incidence: Array of daily case counts (non-negative integers).        serial_interval: SerialInterval distribution for the pathogen.        window: Sliding window size (τ) in days for smoothing.            Larger windows → smoother but more lagged estimates.            Typical values: 7 (default), 14 for noisy data.        prior_shape: Gamma prior shape parameter (a). Default 1.0            gives a weakly informative prior.        prior_scale: Gamma prior scale parameter (b). Default 5.0            gives prior mean = a*b = 5.0 (broad uncertainty).        confidence: Confidence level for credible interval (default 0.95).    Returns:        RtResult with time-varying Rt estimates and credible intervals.    Raises:        ValueError: If incidence contains negative values or is too short.    """    incidence = np.asarray(incidence, dtype=float)    T = len(incidence)    if T < window + 1:        raise ValueError(            f"Incidence series too short: need at least {window + 1} days, "            f"got {T}"        )    if np.any(incidence < 0):        raise ValueError("Incidence values must be non-negative")    # Discretize serial interval    si = serial_interval.discretize(max_days=min(T, 20))    # -----------------------------------------------------------------------    # Compute total infectiousness Λ_t for each day    # Λ_t = Σ_{s=1}^{T} I_{t-s} · w_s    # -----------------------------------------------------------------------    lambda_t = np.zeros(T)    for t in range(1, T):        for s in range(1, min(t + 1, len(si))):            lambda_t[t] += incidence[t - s] * si[s]    # -----------------------------------------------------------------------    # Estimate Rt for each sliding window [t-τ+1, t]    # -----------------------------------------------------------------------    alpha_half = (1.0 - confidence) / 2.0    rt_mean = np.full(T, np.nan)    rt_lower = np.full(T, np.nan)    rt_upper = np.full(T, np.nan)    post_shape = np.full(T, np.nan)    post_scale = np.full(T, np.nan)    for t in range(window, T):        t_start = t - window + 1        t_end = t + 1        # Sum of cases in window        sum_I = np.sum(incidence[t_start:t_end])        # Sum of total infectiousness in window        sum_lambda = np.sum(lambda_t[t_start:t_end])        if sum_lambda == 0:            # Cannot estimate Rt if no prior infectiousness            continue        # Posterior Gamma parameters (conjugate update)        a_post = prior_shape + sum_I        b_post = 1.0 / (1.0 / prior_scale + sum_lambda)        # Store posterior parameters        post_shape[t] = a_post        post_scale[t] = b_post        # Posterior mean        rt_mean[t] = a_post * b_post        # Credible interval from Gamma quantiles        rt_lower[t] = gamma_dist.ppf(alpha_half, a=a_post, scale=b_post)        rt_upper[t] = gamma_dist.ppf(1.0 - alpha_half, a=a_post, scale=b_post)    # -----------------------------------------------------------------------    # Classify epidemic phase at each time step    # -----------------------------------------------------------------------    epidemic_phase = []    for t in range(T):        if np.isnan(rt_mean[t]):            epidemic_phase.append("unknown")        elif rt_lower[t] > 1.0:            # Entire CI above 1 → confidently growing            epidemic_phase.append("growing")        elif rt_upper[t] < 1.0:            # Entire CI below 1 → confidently declining            epidemic_phase.append("declining")        else:            # CI straddles 1 → uncertain, classify as stable            epidemic_phase.append("stable")    result = RtResult(        t=np.arange(T),        rt_mean=rt_mean,        rt_lower=rt_lower,        rt_upper=rt_upper,        epidemic_phase=epidemic_phase,        posterior_shape=post_shape,        posterior_scale=post_scale,    )    logger.info(        "Rt estimation complete: T=%d, current Rt=%.2f (%s), "        "%d valid estimates",        T, result.current_rt, result.current_phase,        int(np.sum(~np.isnan(rt_mean))),    )    return result

### Epi Metrics`epiagent/engines/epi_metrics.py`

In [ ]:
"""Deterministic Epidemiological Metrics Calculator.Computes standard surveillance metrics with proper confidence intervals.All computations are deterministic — no LLM involvement.Metrics:    - Case Fatality Rate (CFR) with Wilson score interval    - Incidence Rate with exact Poisson confidence interval    - Doubling Time via log-linear regression with delta method CI    - Attack Rate with Wilson score interval    - Exponential Growth Rate via log-linear regressionReferences:    Wilson EB. (1927) "Probable Inference, the Law of Succession, and    Statistical Inference." JASA, 22(158):209-212.    Agresti A, Coull BA. (1998) "Approximate is Better than 'Exact' for    Interval Estimation of Binomial Proportions." The American Statistician,    52(2):119-126."""from __future__ import annotationsimport loggingimport warningsfrom dataclasses import dataclassimport numpy as npfrom scipy import statslogger = logging.getLogger(__name__)@dataclassclass MetricResult:    """A single epidemiological metric with confidence interval.    Attributes:        value: Point estimate.        lower_ci: Lower bound of confidence interval.        upper_ci: Upper bound of confidence interval.        method: Description of the CI method used.    """    value: float    lower_ci: float    upper_ci: float    method: str    def summary(self) -> dict:        """Return concise summary dict."""        return {            "value": round(self.value, 6),            "ci": [round(self.lower_ci, 6), round(self.upper_ci, 6)],            "method": self.method,        }# ---------------------------------------------------------------------------# Wilson Score Interval (helper)# ---------------------------------------------------------------------------def _wilson_score_interval(    successes: int,    trials: int,    confidence: float = 0.95,) -> tuple[float, float, float]:    """Compute Wilson score interval for a binomial proportion.    The Wilson score interval has better coverage properties than the    Wald interval, especially for small n or extreme p.    Formula:        CI = (p̂ + z²/2n ± z·√(p̂(1-p̂)/n + z²/4n²)) / (1 + z²/n)    Args:        successes: Number of successes (e.g., deaths).        trials: Number of trials (e.g., cases).        confidence: Confidence level (default 0.95).    Returns:        Tuple of (point_estimate, lower_ci, upper_ci).    """    if trials <= 0:        return (float("nan"), float("nan"), float("nan"))    p_hat = successes / trials    z = stats.norm.ppf(1 - (1 - confidence) / 2)    z2 = z ** 2    denominator = 1 + z2 / trials    center = p_hat + z2 / (2 * trials)    margin = z * np.sqrt(p_hat * (1 - p_hat) / trials + z2 / (4 * trials ** 2))    lower = (center - margin) / denominator    upper = (center + margin) / denominator    # Clamp to [0, 1]    lower = max(0.0, lower)    upper = min(1.0, upper)    return (p_hat, lower, upper)# ---------------------------------------------------------------------------# Public API# ---------------------------------------------------------------------------def compute_cfr(    deaths: int,    cases: int,    confidence: float = 0.95,) -> MetricResult:    """Compute Case Fatality Rate with Wilson score confidence interval.    CFR = deaths / cases    Uses the Wilson score interval which has proper coverage even for    small sample sizes and extreme proportions — a significant improvement    over the Wald interval (p̂ ± z√(p̂(1-p̂)/n)) which fails at boundaries.    Args:        deaths: Number of deaths.        cases: Number of cases (denominator).        confidence: Confidence level (default 0.95).    Returns:        MetricResult with CFR and Wilson score CI.    Raises:        ValueError: If deaths or cases are negative.    """    if deaths < 0 or cases < 0:        raise ValueError(f"Counts must be non-negative: deaths={deaths}, cases={cases}")    if cases == 0:        logger.warning("CFR undefined: 0 cases. Returning NaN.")        return MetricResult(            value=float("nan"),            lower_ci=float("nan"),            upper_ci=float("nan"),            method="Wilson score (undefined: 0 cases)",        )    if deaths > cases:        logger.warning(            "Deaths (%d) exceed cases (%d). CFR will exceed 1.0. "            "This may indicate reporting lag or data error.",            deaths, cases,        )    cfr, lower, upper = _wilson_score_interval(deaths, cases, confidence)    return MetricResult(        value=cfr,        lower_ci=lower,        upper_ci=upper,        method=f"Wilson score interval ({confidence:.0%} CI)",    )def compute_incidence_rate(    cases: int,    population: int,    per: int = 100_000,    confidence: float = 0.95,) -> MetricResult:    """Compute incidence rate with exact Poisson confidence interval.    Rate = (cases / population) × per    Uses the exact Poisson CI based on the chi-squared distribution:        Lower = χ²(α/2, 2k) / (2n)        Upper = χ²(1-α/2, 2k+2) / (2n)    where k = cases and n = population/per.    Args:        cases: Number of new cases.        population: Population at risk.        per: Rate multiplier (default 100,000 for "per 100k").        confidence: Confidence level (default 0.95).    Returns:        MetricResult with incidence rate and exact Poisson CI.    Raises:        ValueError: If cases is negative or population is non-positive.    """    if cases < 0:        raise ValueError(f"Cases must be non-negative: {cases}")    if population <= 0:        raise ValueError(f"Population must be positive: {population}")    rate = (cases / population) * per    alpha = 1 - confidence    if cases == 0:        lower = 0.0        upper = (stats.chi2.ppf(1 - alpha / 2, 2) / (2 * population)) * per    else:        lower = (stats.chi2.ppf(alpha / 2, 2 * cases) / (2 * population)) * per        upper = (stats.chi2.ppf(1 - alpha / 2, 2 * (cases + 1)) / (2 * population)) * per    return MetricResult(        value=rate,        lower_ci=lower,        upper_ci=upper,        method=f"Exact Poisson CI ({confidence:.0%}), per {per:,}",    )def compute_doubling_time(    case_series: np.ndarray,    window: int = 7,    confidence: float = 0.95,) -> MetricResult:    """Compute epidemic doubling time via log-linear regression.    Fits log(cases) = a + r·t to the most recent `window` days,    then Td = ln(2) / r.    The confidence interval uses the delta method:        SE(Td) = ln(2) / r² · SE(r)    Args:        case_series: Array of daily case counts.        window: Number of recent days to use for fitting (default 7).        confidence: Confidence level (default 0.95).    Returns:        MetricResult with doubling time in days.        Returns NaN if growth rate is non-positive (epidemic declining).    """    series = np.asarray(case_series, dtype=float)    if len(series) < window:        return MetricResult(            value=float("nan"),            lower_ci=float("nan"),            upper_ci=float("nan"),            method=f"Log-linear regression (insufficient data: {len(series)} < {window})",        )    # Use last `window` days    recent = series[-window:]    # Filter out zeros and negative values for log transform    valid_mask = recent > 0    if valid_mask.sum() < 3:        return MetricResult(            value=float("nan"),            lower_ci=float("nan"),            upper_ci=float("nan"),            method="Log-linear regression (insufficient positive values)",        )    t = np.arange(window)[valid_mask]    y = np.log(recent[valid_mask])    # Linear regression: log(cases) = intercept + r * t    result = stats.linregress(t, y)    r = result.slope    se_r = result.stderr    if r <= 0:        # Epidemic is declining — doubling time is not meaningful        return MetricResult(            value=float("nan"),            lower_ci=float("nan"),            upper_ci=float("nan"),            method="Log-linear regression (growth rate ≤ 0, epidemic declining)",        )    # Doubling time    Td = np.log(2) / r    # Delta method CI for Td = ln(2)/r    # SE(Td) = ln(2) / r² · SE(r)    z = stats.norm.ppf(1 - (1 - confidence) / 2)    se_Td = np.log(2) / (r ** 2) * se_r    lower = max(0.0, Td - z * se_Td)    upper = Td + z * se_Td    return MetricResult(        value=Td,        lower_ci=lower,        upper_ci=upper,        method=f"Log-linear regression, delta method ({confidence:.0%} CI), "               f"window={window} days, r={r:.4f}",    )def compute_attack_rate(    total_cases: int,    population: int,    confidence: float = 0.95,) -> MetricResult:    """Compute attack rate (cumulative incidence proportion) with Wilson CI.    Attack Rate = total_cases / population    Args:        total_cases: Total cumulative cases.        population: Population at risk.        confidence: Confidence level (default 0.95).    Returns:        MetricResult with attack rate and Wilson score CI.    Raises:        ValueError: If inputs are invalid.    """    if total_cases < 0:        raise ValueError(f"Total cases must be non-negative: {total_cases}")    if population <= 0:        raise ValueError(f"Population must be positive: {population}")    ar, lower, upper = _wilson_score_interval(total_cases, population, confidence)    return MetricResult(        value=ar,        lower_ci=lower,        upper_ci=upper,        method=f"Wilson score interval ({confidence:.0%} CI)",    )def compute_growth_rate(    case_series: np.ndarray,    window: int = 7,    confidence: float = 0.95,) -> MetricResult:    """Compute exponential growth rate via log-linear regression.    Fits log(cases) = a + r·t to the most recent `window` days.    Positive r → exponential growth; negative r → exponential decay.    Args:        case_series: Array of daily case counts.        window: Number of recent days to use (default 7).        confidence: Confidence level (default 0.95).    Returns:        MetricResult with growth rate r (per day).    """    series = np.asarray(case_series, dtype=float)    if len(series) < window:        return MetricResult(            value=float("nan"),            lower_ci=float("nan"),            upper_ci=float("nan"),            method=f"Log-linear regression (insufficient data: {len(series)} < {window})",        )    recent = series[-window:]    valid_mask = recent > 0    if valid_mask.sum() < 3:        return MetricResult(            value=float("nan"),            lower_ci=float("nan"),            upper_ci=float("nan"),            method="Log-linear regression (insufficient positive values)",        )    t = np.arange(window)[valid_mask]    y = np.log(recent[valid_mask])    result = stats.linregress(t, y)    r = result.slope    se_r = result.stderr    z = stats.norm.ppf(1 - (1 - confidence) / 2)    lower = r - z * se_r    upper = r + z * se_r    return MetricResult(        value=r,        lower_ci=lower,        upper_ci=upper,        method=f"Log-linear regression ({confidence:.0%} CI), window={window} days",    )

### Changepoint Detector`epiagent/engines/changepoint_detector.py`

In [ ]:
"""Bayesian Online Changepoint Detection (BOCPD).Implements the Adams & MacKay (2007) algorithm for detecting structuralbreaks in epidemic time series — outbreak onset, peak, decline, andresurgence.Reference:    Adams RP, MacKay DJC. (2007)    "Bayesian Online Changepoint Detection."    arXiv:0710.3742Method:    Maintains a posterior distribution over the "run length" r_t    (number of time steps since the last changepoint). At each    new observation x_t:        1. Compute predictive probability P(x_t | r_{t-1}) using a       conjugate Normal-Inverse-Gamma model    2. Update growth probabilities (r_t = r_{t-1} + 1)    3. Update changepoint probability (r_t = 0)    4. Normalize to get posterior P(r_t | x_{1:t})    5. P(changepoint at t) = P(r_t = 0 | x_{1:t})    Uses a constant hazard function H = 1/λ where λ is the    expected run length between changepoints."""from __future__ import annotationsimport loggingfrom dataclasses import dataclass, fieldimport numpy as npfrom scipy.stats import t as student_tlogger = logging.getLogger(__name__)@dataclassclass ChangePointResult:    """Result of Bayesian online changepoint detection.    Attributes:        changepoint_probs: Array of changepoint probability at each time step.        changepoints: Indices where changepoint probability exceeds threshold.        run_length_posterior: Full run length posterior matrix R[t, r].        threshold: Threshold used for changepoint identification.    """    changepoint_probs: np.ndarray    changepoints: list[int]    run_length_posterior: np.ndarray    threshold: float    def summary(self) -> dict:        """Return concise summary for agent state."""        return {            "n_changepoints": len(self.changepoints),            "changepoint_indices": self.changepoints,            "max_changepoint_prob": float(np.nanmax(self.changepoint_probs))            if len(self.changepoint_probs) > 0            else 0.0,            "threshold": self.threshold,        }class BOCPDetector:    """Bayesian Online Changepoint Detector.    Uses a Normal-Inverse-Gamma conjugate prior for the observation model,    meaning it assumes observations are Gaussian with unknown mean and variance.    For epidemic case counts, it's recommended to apply a log-transform    or Box-Cox transform before running detection to better approximate    normality.    Args:        hazard_lambda: Expected run length between changepoints.            Smaller values make the detector more sensitive.            Typical values: 50-250 for daily data.        mu0: Prior mean of the Normal-Inverse-Gamma distribution.        kappa0: Prior precision scaling (higher = more confident in mu0).        alpha0: Prior shape of the Inverse-Gamma (must be > 0).        beta0: Prior scale of the Inverse-Gamma (must be > 0).    """    def __init__(        self,        hazard_lambda: float = 100.0,        mu0: float = 0.0,        kappa0: float = 1.0,        alpha0: float = 1.0,        beta0: float = 1.0,    ):        if hazard_lambda <= 0:            raise ValueError(f"hazard_lambda must be > 0, got {hazard_lambda}")        if alpha0 <= 0 or beta0 <= 0:            raise ValueError(f"alpha0 and beta0 must be > 0, got alpha0={alpha0}, beta0={beta0}")        self.hazard = 1.0 / hazard_lambda        self.mu0 = mu0        self.kappa0 = kappa0        self.alpha0 = alpha0        self.beta0 = beta0    def detect(        self,        data: np.ndarray,        threshold: float = 0.5,    ) -> ChangePointResult:        """Run BOCPD on a 1-D time series.        Args:            data: 1-D array of observations (e.g., daily case counts,                  ideally log-transformed).            threshold: Probability threshold for declaring a changepoint.                       Lower values = more sensitive detection.        Returns:            ChangePointResult with probabilities and detected changepoints.        """        data = np.asarray(data, dtype=float)        T = len(data)        if T == 0:            return ChangePointResult(                changepoint_probs=np.array([]),                changepoints=[],                run_length_posterior=np.zeros((1, 1)),                threshold=threshold,            )        # Run length posterior: R[t, r] = P(r_t = r | x_{1:t})        # We only need the current and previous time step for memory efficiency,        # but we store the full matrix for visualization purposes.        R = np.zeros((T + 1, T + 1))        R[0, 0] = 1.0  # Prior: run length 0 with probability 1        # Sufficient statistics for each run length hypothesis        # Normal-Inverse-Gamma: track (mu, kappa, alpha, beta)        mu = np.array([self.mu0])        kappa = np.array([self.kappa0])        alpha = np.array([self.alpha0])        beta = np.array([self.beta0])        changepoint_probs = np.zeros(T)        for t in range(T):            x = data[t]            # ---------------------------------------------------------------            # 1. Predictive probability under each run length hypothesis            #    Using Student-t distribution (conjugate predictive)            # ---------------------------------------------------------------            df = 2.0 * alpha            loc = mu            scale = np.sqrt(beta * (kappa + 1.0) / (alpha * kappa))            # Clamp scale to avoid numerical issues            scale = np.maximum(scale, 1e-10)            pred_probs = student_t.pdf(x, df=df, loc=loc, scale=scale)            # Clamp to avoid zero probabilities            pred_probs = np.maximum(pred_probs, 1e-300)            # ---------------------------------------------------------------            # 2. Growth probabilities: r_t = r_{t-1} + 1            # ---------------------------------------------------------------            growth = R[t, : t + 1] * pred_probs * (1.0 - self.hazard)            # ---------------------------------------------------------------            # 3. Changepoint probability: r_t = 0            # ---------------------------------------------------------------            cp = np.sum(R[t, : t + 1] * pred_probs * self.hazard)            # ---------------------------------------------------------------            # 4. Update run length distribution            # ---------------------------------------------------------------            R[t + 1, 0] = cp            R[t + 1, 1 : t + 2] = growth            # ---------------------------------------------------------------            # 5. Normalize            # ---------------------------------------------------------------            evidence = R[t + 1, : t + 2].sum()            if evidence > 0:                R[t + 1, : t + 2] /= evidence            else:                # Fallback: reset to uniform if numerical underflow                R[t + 1, 0] = 1.0            changepoint_probs[t] = R[t + 1, 0]            # ---------------------------------------------------------------            # 6. Update sufficient statistics for each run length            # ---------------------------------------------------------------            new_kappa = np.append(self.kappa0, kappa + 1.0)            new_alpha = np.append(self.alpha0, alpha + 0.5)            new_mu = np.append(                self.mu0,                (kappa * mu + x) / (kappa + 1.0),            )            new_beta = np.append(                self.beta0,                beta + kappa * (x - mu) ** 2 / (2.0 * (kappa + 1.0)),            )            mu, kappa, alpha, beta = new_mu, new_kappa, new_alpha, new_beta        # ---------------------------------------------------------------        # Identify changepoints above threshold        # ---------------------------------------------------------------        changepoints = list(np.where(changepoint_probs > threshold)[0])        logger.info(            "BOCPD complete: T=%d, detected %d changepoints (threshold=%.2f)",            T, len(changepoints), threshold,        )        return ChangePointResult(            changepoint_probs=changepoint_probs,            changepoints=changepoints,            run_length_posterior=R,            threshold=threshold,        )def detect_outbreak_signals(    case_series: np.ndarray,    *,    hazard_lambda: float = 100.0,    threshold: float = 0.5,    log_transform: bool = True,) -> ChangePointResult:    """Convenience function for outbreak signal detection in case counts.    Applies optional log-transform to better approximate normality    (case counts are typically right-skewed), then runs BOCPD.    Args:        case_series: Array of daily case counts.        hazard_lambda: Expected run length between changepoints.        threshold: Changepoint probability threshold.        log_transform: If True, apply log(1 + x) transform before detection.    Returns:        ChangePointResult with detected outbreak signals.    """    case_series = np.asarray(case_series, dtype=float)    if log_transform:        # log(1 + x) handles zeros and makes distribution more Gaussian        transformed = np.log1p(np.maximum(case_series, 0))    else:        transformed = case_series.copy()    # Replace NaN/Inf with 0    transformed = np.nan_to_num(transformed, nan=0.0, posinf=0.0, neginf=0.0)    detector = BOCPDetector(hazard_lambda=hazard_lambda)    return detector.detect(transformed, threshold=threshold)

### ML Forecaster`epiagent/engines/ml_forecaster.py`

In [ ]:
"""ML Forecasting Ensemble for Epidemic Time Series.Implements a two-model ensemble for epidemic case forecasting:1. XGBoost Gradient Boosted Trees — Feature-engineered supervised model   using lag features, rolling statistics, and epidemiological covariates.   Provides the primary forecast with SHAP explainability.2. Prophet (Meta/Facebook) — Additive decomposition model with   trend + seasonality + holiday components. Optional dependency.Ensemble Strategy:    Weighted average using inverse RMSE from validation set.    If only one model is available, uses that model alone.The forecaster is designed to be called as a deterministic FunctionToolby the ML Forecasting Agent — the LLM interprets results, not computes them."""from __future__ import annotationsimport loggingimport warningsfrom dataclasses import dataclass, fieldfrom datetime import datetime, timedeltaimport numpy as npimport pandas as pdfrom sklearn.model_selection import TimeSeriesSplitfrom sklearn.metrics import mean_squared_error, mean_absolute_errorlogger = logging.getLogger(__name__)@dataclassclass ForecastOutput:    """Output from a single forecasting model.    Attributes:        dates: Forecast date strings (ISO 8601).        predicted: Point predictions.        lower_bound: Lower prediction interval (approx 95%).        upper_bound: Upper prediction interval (approx 95%).        model_name: Name of the model.        rmse: Root Mean Squared Error on validation data.        mae: Mean Absolute Error on validation data.        feature_importance: Feature importance dict (for tree models).    """    dates: list[str]    predicted: list[float]    lower_bound: list[float]    upper_bound: list[float]    model_name: str    rmse: float | None = None    mae: float | None = None    feature_importance: dict[str, float] = field(default_factory=dict)    def to_dict(self) -> dict:        """Convert to JSON-serializable dict."""        return {            "dates": self.dates,            "predicted": [round(p, 2) for p in self.predicted],            "lower_bound": [round(lb, 2) for lb in self.lower_bound],            "upper_bound": [round(ub, 2) for ub in self.upper_bound],            "model_name": self.model_name,            "rmse": round(self.rmse, 4) if self.rmse else None,            "mae": round(self.mae, 4) if self.mae else None,            "feature_importance": {                k: round(v, 4) for k, v in self.feature_importance.items()            },        }# ---------------------------------------------------------------------------# Feature Engineering# ---------------------------------------------------------------------------def create_lag_features(    case_series: np.ndarray,    dates: list[str] | None = None,    n_lags: int = 14,) -> pd.DataFrame:    """Create supervised learning features from epidemic time series.    Features:        - Lag features: cases at t-1, t-2, ..., t-n_lags        - Rolling statistics: 7-day and 14-day mean, std, min, max        - Growth features: day-over-day change, 7-day change ratio        - Calendar features: day of week, month, is_weekend    Args:        case_series: Array of daily case counts.        dates: Optional list of ISO date strings.        n_lags: Number of lag features to create.    Returns:        DataFrame with features and 'target' column.    """    n = len(case_series)    df = pd.DataFrame({"cases": case_series.astype(float)})    if dates is not None:        df["date"] = pd.to_datetime(dates)    else:        df["date"] = pd.date_range(end=datetime.today(), periods=n, freq="D")    # Lag features    for lag in range(1, n_lags + 1):        df[f"lag_{lag}"] = df["cases"].shift(lag)    # Rolling statistics (shifted by 1 to prevent data leakage)    for window in [7, 14]:        rolling = df["cases"].shift(1).rolling(window=window, min_periods=1)        df[f"rolling_{window}d_mean"] = rolling.mean()        df[f"rolling_{window}d_std"] = rolling.std().fillna(0)        df[f"rolling_{window}d_min"] = rolling.min()        df[f"rolling_{window}d_max"] = rolling.max()    # Growth features    df["day_change"] = df["cases"].diff().shift(1)    prev_7d = df["cases"].shift(7)    df["week_change_ratio"] = np.where(        prev_7d > 0,        df["cases"].shift(1) / prev_7d,        1.0,    )    # Calendar features    df["day_of_week"] = df["date"].dt.dayofweek    df["month"] = df["date"].dt.month    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)    # Target    df["target"] = df["cases"]    # Drop rows with NaN from lagging    df = df.dropna().reset_index(drop=True)    return df# ---------------------------------------------------------------------------# XGBoost Forecaster# ---------------------------------------------------------------------------def forecast_xgboost(    case_series: np.ndarray,    dates: list[str] | None = None,    horizon: int = 14,    n_lags: int = 14,    test_size: int = 14,) -> ForecastOutput:    """Forecast epidemic cases using XGBoost with engineered features.    Args:        case_series: Array of daily case counts.        dates: Optional list of ISO date strings.        horizon: Number of days to forecast ahead.        n_lags: Number of lag features.        test_size: Number of recent days for validation.    Returns:        ForecastOutput with predictions and feature importance.    """    try:        import xgboost as xgb    except ImportError:        logger.error("XGBoost not installed. Run: pip install xgboost")        raise    # Create features    df = create_lag_features(case_series, dates, n_lags=n_lags)    if len(df) < test_size + 10:        raise ValueError(            f"Insufficient data for XGBoost: need at least "            f"{test_size + 10} rows after feature engineering, got {len(df)}"        )    # Feature columns (exclude date, target, cases)    feature_cols = [        c for c in df.columns        if c not in ("date", "target", "cases")    ]    # Train/validation split (temporal)    train_df = df.iloc[:-test_size]    val_df = df.iloc[-test_size:]    X_train = train_df[feature_cols].values    y_train = train_df["target"].values    X_val = val_df[feature_cols].values    y_val = val_df["target"].values    # Train XGBoost    model = xgb.XGBRegressor(        n_estimators=200,        max_depth=6,        learning_rate=0.1,        subsample=0.8,        colsample_bytree=0.8,        min_child_weight=3,        random_state=42,        verbosity=0,    )    with warnings.catch_warnings():        warnings.simplefilter("ignore")        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)    # Validation metrics    val_pred = model.predict(X_val)    rmse = float(np.sqrt(mean_squared_error(y_val, val_pred)))    mae = float(mean_absolute_error(y_val, val_pred))    # Feature importance    importance = dict(zip(feature_cols, model.feature_importances_.tolist()))    # Sort by importance    importance = dict(        sorted(importance.items(), key=lambda x: x[1], reverse=True)[:15]    )    # Recursive multi-step forecast    last_known = df.iloc[-1].copy()    predictions = []    last_date = df["date"].iloc[-1]    for step in range(horizon):        # Build feature vector for next day        feat_dict = {}        # Update lags: shift everything forward        for lag in range(n_lags, 1, -1):            feat_dict[f"lag_{lag}"] = last_known.get(f"lag_{lag - 1}", 0)        feat_dict["lag_1"] = last_known.get("target", last_known.get("cases", 0))        # Rolling stats (approximate using recent predictions)        recent_cases = list(case_series[-13:]) + predictions        recent = np.array(recent_cases[-14:], dtype=float)        for window in [7, 14]:            w = min(window, len(recent))            feat_dict[f"rolling_{window}d_mean"] = float(np.mean(recent[-w:]))            feat_dict[f"rolling_{window}d_std"] = float(np.std(recent[-w:])) if w > 1 else 0.0            feat_dict[f"rolling_{window}d_min"] = float(np.min(recent[-w:]))            feat_dict[f"rolling_{window}d_max"] = float(np.max(recent[-w:]))        # Growth features        if len(predictions) > 0:            feat_dict["day_change"] = predictions[-1] - (                predictions[-2] if len(predictions) > 1 else case_series[-1]            )        else:            feat_dict["day_change"] = float(np.diff(case_series[-2:]).item()) if len(case_series) > 1 else 0.0        prev_week = recent[-7] if len(recent) >= 7 else recent[0]        feat_dict["week_change_ratio"] = (            recent[-1] / prev_week if prev_week > 0 else 1.0        )        # Calendar features        next_date = last_date + timedelta(days=step + 1)        feat_dict["day_of_week"] = next_date.weekday()        feat_dict["month"] = next_date.month        feat_dict["is_weekend"] = 1 if next_date.weekday() in (5, 6) else 0        # Predict        X_next = np.array([[feat_dict.get(c, 0) for c in feature_cols]])        pred = float(model.predict(X_next)[0])        pred = max(0.0, pred)  # Non-negative cases        predictions.append(pred)        # Update last_known for next step        for k, v in feat_dict.items():            last_known[k] = v        last_known["target"] = pred        last_known["cases"] = pred    # Generate forecast dates    forecast_dates = [        (last_date + timedelta(days=i + 1)).strftime("%Y-%m-%d")        for i in range(horizon)    ]    # Uncertainty estimation (naive: based on validation residuals)    val_residuals_std = float(np.std(y_val - val_pred))    lower = [max(0, p - 1.96 * val_residuals_std) for p in predictions]    upper = [p + 1.96 * val_residuals_std for p in predictions]    return ForecastOutput(        dates=forecast_dates,        predicted=predictions,        lower_bound=lower,        upper_bound=upper,        model_name="XGBoost",        rmse=rmse,        mae=mae,        feature_importance=importance,    )# ---------------------------------------------------------------------------# Prophet Forecaster (optional dependency)# ---------------------------------------------------------------------------def forecast_prophet(    case_series: np.ndarray,    dates: list[str] | None = None,    horizon: int = 14,) -> ForecastOutput | None:    """Forecast using Facebook Prophet (optional dependency).    Args:        case_series: Array of daily case counts.        dates: Optional list of ISO date strings.        horizon: Forecast horizon in days.    Returns:        ForecastOutput or None if Prophet is not installed.    """    try:        from prophet import Prophet    except ImportError:        logger.warning(            "Prophet not installed. Skipping Prophet forecast. "            "Install with: pip install prophet"        )        return None    # Prepare Prophet dataframe    if dates is not None:        ds = pd.to_datetime(dates)    else:        ds = pd.date_range(end=datetime.today(), periods=len(case_series), freq="D")    df = pd.DataFrame({"ds": ds, "y": case_series.astype(float)})    # Remove zeros/negatives for better Prophet fit    df["y"] = df["y"].clip(lower=0)    # Validation: hold out last 14 days    train_df = df.iloc[:-14] if len(df) > 28 else df    val_df = df.iloc[-14:] if len(df) > 28 else None    # Fit Prophet    with warnings.catch_warnings():        warnings.simplefilter("ignore")        model = Prophet(            growth="linear",            seasonality_mode="multiplicative",            changepoint_prior_scale=0.05,            yearly_seasonality=False,            weekly_seasonality=True,            daily_seasonality=False,        )        model.fit(train_df)    # Validation metrics    rmse = None    mae = None    if val_df is not None:        val_pred = model.predict(val_df[["ds"]])        val_y = val_df["y"].values        val_yhat = val_pred["yhat"].values        rmse = float(np.sqrt(mean_squared_error(val_y, val_yhat)))        mae = float(mean_absolute_error(val_y, val_yhat))    # Forecast    future = model.make_future_dataframe(periods=horizon)    forecast = model.predict(future)    # Extract forecast period    fc = forecast.iloc[-horizon:]    forecast_dates = fc["ds"].dt.strftime("%Y-%m-%d").tolist()    predicted = fc["yhat"].clip(lower=0).tolist()    lower = fc["yhat_lower"].clip(lower=0).tolist()    upper = fc["yhat_upper"].clip(lower=0).tolist()    return ForecastOutput(        dates=forecast_dates,        predicted=predicted,        lower_bound=lower,        upper_bound=upper,        model_name="Prophet",        rmse=rmse,        mae=mae,    )# ---------------------------------------------------------------------------# Ensemble Forecaster# ---------------------------------------------------------------------------def forecast_ensemble(    case_series: np.ndarray,    dates: list[str] | None = None,    horizon: int = 14,    n_lags: int = 14,) -> tuple[ForecastOutput, list[ForecastOutput]]:    """Run ensemble forecast combining XGBoost and Prophet.    Uses inverse-RMSE weighting: models with lower validation RMSE    get higher weight in the ensemble.    Args:        case_series: Array of daily case counts.        dates: Optional list of ISO date strings.        horizon: Forecast horizon in days.        n_lags: Number of lag features for XGBoost.    Returns:        Tuple of (ensemble_forecast, list_of_individual_forecasts).    """    individual_forecasts: list[ForecastOutput] = []    # XGBoost (always available)    try:        xgb_forecast = forecast_xgboost(            case_series, dates, horizon=horizon, n_lags=n_lags,        )        individual_forecasts.append(xgb_forecast)        logger.info("XGBoost forecast complete: RMSE=%.2f", xgb_forecast.rmse or 0)    except Exception as e:        logger.error("XGBoost forecast failed: %s", e)    # Prophet (optional)    try:        prophet_forecast = forecast_prophet(case_series, dates, horizon=horizon)        if prophet_forecast is not None:            individual_forecasts.append(prophet_forecast)            logger.info("Prophet forecast complete: RMSE=%.2f", prophet_forecast.rmse or 0)    except Exception as e:        logger.warning("Prophet forecast failed: %s", e)    if not individual_forecasts:        raise RuntimeError("All forecasting models failed")    # Single model → return directly    if len(individual_forecasts) == 1:        fc = individual_forecasts[0]        ensemble = ForecastOutput(            dates=fc.dates,            predicted=fc.predicted,            lower_bound=fc.lower_bound,            upper_bound=fc.upper_bound,            model_name=f"Ensemble ({fc.model_name} only)",            rmse=fc.rmse,            mae=fc.mae,            feature_importance=fc.feature_importance,        )        return ensemble, individual_forecasts    # Inverse-RMSE weighting    rmses = []    for fc in individual_forecasts:        rmses.append(fc.rmse if fc.rmse and fc.rmse > 0 else 1e6)    inv_rmses = [1.0 / r for r in rmses]    total_weight = sum(inv_rmses)    weights = [w / total_weight for w in inv_rmses]    logger.info(        "Ensemble weights: %s",        {fc.model_name: f"{w:.2%}" for fc, w in zip(individual_forecasts, weights)},    )    # Weighted average    n = len(individual_forecasts[0].dates)    ensemble_pred = np.zeros(n)    ensemble_lower = np.zeros(n)    ensemble_upper = np.zeros(n)    for fc, w in zip(individual_forecasts, weights):        pred = np.array(fc.predicted[:n])        lower = np.array(fc.lower_bound[:n])        upper = np.array(fc.upper_bound[:n])        ensemble_pred += w * pred        ensemble_lower += w * lower        ensemble_upper += w * upper    # Ensemble RMSE (weighted average of individual RMSEs)    ensemble_rmse = sum(w * r for w, r in zip(weights, rmses))    ensemble = ForecastOutput(        dates=individual_forecasts[0].dates,        predicted=ensemble_pred.tolist(),        lower_bound=ensemble_lower.tolist(),        upper_bound=ensemble_upper.tolist(),        model_name=f"Ensemble ({'+'.join(fc.model_name for fc in individual_forecasts)})",        rmse=ensemble_rmse,        feature_importance=individual_forecasts[0].feature_importance,  # from XGBoost    )    return ensemble, individual_forecasts

### SHAP Explainability`epiagent/engines/explainability.py`

In [ ]:
"""SHAP Explainability Module for Epidemic Forecasting Models.Provides post-hoc explainability for XGBoost epidemic forecasts usingSHAP (SHapley Additive exPlanations) values.SHAP provides:    1. Global feature importance — which features matter most overall    2. Local explanations — why was this specific prediction made    3. Feature interaction effects — how features combine    4. Temporal attribution — how driver importance changes over timeThis module wraps SHAP's TreeExplainer for fast, exact computationof Shapley values for tree-based models (XGBoost, Random Forest).References:    Lundberg SM, Lee SI. (2017) "A Unified Approach to Interpreting    Model Predictions." NeurIPS."""from __future__ import annotationsimport loggingfrom dataclasses import dataclass, fieldimport numpy as npimport pandas as pdlogger = logging.getLogger(__name__)@dataclassclass SHAPExplanation:    """SHAP analysis results for a forecast model.    Attributes:        feature_names: List of feature names.        global_importance: Mean |SHAP| for each feature (global ranking).        shap_values: Full SHAP value matrix (n_samples × n_features).        top_drivers: Top-k most important features with their scores.        base_value: Expected model output (baseline prediction).        temporal_importance: Feature importance over time windows.    """    feature_names: list[str]    global_importance: dict[str, float]    shap_values: np.ndarray    top_drivers: list[dict[str, float]]    base_value: float    temporal_importance: dict[str, list[float]] = field(default_factory=dict)    def to_dict(self) -> dict:        """Convert to JSON-serializable dict (excluding raw SHAP matrix)."""        return {            "top_drivers": self.top_drivers,            "global_importance": {                k: round(v, 4) for k, v in self.global_importance.items()            },            "base_value": round(self.base_value, 2),            "temporal_importance": {                k: [round(x, 4) for x in v]                for k, v in self.temporal_importance.items()            },        }    def narrative(self) -> str:        """Generate human-readable narrative of SHAP findings.        Returns:            Multi-sentence explanation suitable for SitRep inclusion.        """        if not self.top_drivers:            return "Insufficient data for SHAP analysis."        lines = ["**Forecast Driver Analysis (SHAP):**"]        for i, driver in enumerate(self.top_drivers[:5], 1):            name = driver["feature"]            importance = driver["importance"]            # Translate feature names to epidemiological terms            epi_name = _translate_feature_name(name)            lines.append(                f"{i}. **{epi_name}** (importance: {importance:.3f})"            )        # Add temporal trend if available        if self.temporal_importance:            top_feat = self.top_drivers[0]["feature"]            if top_feat in self.temporal_importance:                recent = self.temporal_importance[top_feat][-3:]                if len(recent) >= 2:                    trend = "increasing" if recent[-1] > recent[0] else "decreasing"                    lines.append(                        f"\nThe influence of {_translate_feature_name(top_feat)} "                        f"has been **{trend}** over recent time windows."                    )        return "\n".join(lines)def _translate_feature_name(name: str) -> str:    """Translate ML feature names to epidemiological terms."""    translations = {        "lag_1": "yesterday's case count",        "lag_2": "cases 2 days ago",        "lag_7": "cases 1 week ago",        "lag_14": "cases 2 weeks ago",        "rolling_7d_mean": "7-day average case count",        "rolling_7d_std": "7-day case count variability",        "rolling_14d_mean": "14-day average case count",        "rolling_14d_std": "14-day case count variability",        "rolling_7d_min": "7-day minimum daily cases",        "rolling_7d_max": "7-day peak daily cases",        "rolling_14d_min": "14-day minimum daily cases",        "rolling_14d_max": "14-day peak daily cases",        "day_change": "day-over-day change in cases",        "week_change_ratio": "week-over-week growth ratio",        "day_of_week": "day of the week (reporting pattern)",        "month": "month of year (seasonality)",        "is_weekend": "weekend reporting effect",    }    return translations.get(name, name)def explain_forecast(    model,    X_data: np.ndarray,    feature_names: list[str],    top_k: int = 10,    n_temporal_windows: int = 4,) -> SHAPExplanation:    """Compute SHAP explanations for an XGBoost forecast model.    Args:        model: Trained XGBoost model (or any tree-based model).        X_data: Feature matrix (n_samples × n_features).        feature_names: List of feature names.        top_k: Number of top features to highlight.        n_temporal_windows: Number of time windows for temporal analysis.    Returns:        SHAPExplanation with global and local explanations.    """    try:        import shap    except ImportError:        logger.warning(            "SHAP not installed. Falling back to model feature_importances_."        )        return _fallback_importance(model, feature_names, top_k)    # TreeExplainer for exact, fast Shapley values    explainer = shap.TreeExplainer(model)    shap_values = explainer.shap_values(X_data)    # Base value    base_value = float(explainer.expected_value)    # Global importance: mean |SHAP| per feature    mean_abs_shap = np.abs(shap_values).mean(axis=0)    global_importance = dict(zip(feature_names, mean_abs_shap.tolist()))    # Sort by importance    sorted_features = sorted(        global_importance.items(), key=lambda x: x[1], reverse=True    )    # Top-k drivers    top_drivers = [        {"feature": name, "importance": score}        for name, score in sorted_features[:top_k]    ]    # Temporal importance: split data into windows and compute per-window SHAP    temporal_importance = {}    if len(X_data) >= n_temporal_windows * 5:        window_size = len(X_data) // n_temporal_windows        top_feature_names = [d["feature"] for d in top_drivers[:5]]        for feat_name in top_feature_names:            feat_idx = feature_names.index(feat_name)            window_importances = []            for w in range(n_temporal_windows):                start = w * window_size                end = (w + 1) * window_size if w < n_temporal_windows - 1 else len(X_data)                window_shap = np.abs(shap_values[start:end, feat_idx]).mean()                window_importances.append(float(window_shap))            temporal_importance[feat_name] = window_importances    return SHAPExplanation(        feature_names=feature_names,        global_importance=dict(sorted_features),        shap_values=shap_values,        top_drivers=top_drivers,        base_value=base_value,        temporal_importance=temporal_importance,    )def _fallback_importance(    model,    feature_names: list[str],    top_k: int,) -> SHAPExplanation:    """Fallback when SHAP is not available: use model's feature_importances_.    Args:        model: Tree model with feature_importances_ attribute.        feature_names: Feature names.        top_k: Number of top features.    Returns:        SHAPExplanation with feature importance (no SHAP values).    """    if hasattr(model, "feature_importances_"):        importances = model.feature_importances_.tolist()    else:        logger.warning("Model has no feature_importances_. Using uniform.")        importances = [1.0 / len(feature_names)] * len(feature_names)    global_importance = dict(zip(feature_names, importances))    sorted_features = sorted(        global_importance.items(), key=lambda x: x[1], reverse=True    )    top_drivers = [        {"feature": name, "importance": score}        for name, score in sorted_features[:top_k]    ]    return SHAPExplanation(        feature_names=feature_names,        global_importance=dict(sorted_features),        shap_values=np.array([]),        top_drivers=top_drivers,        base_value=0.0,    )def explain_xgboost_forecast(    case_series: np.ndarray,    dates: list[str] | None = None,    n_lags: int = 14,    top_k: int = 10,) -> tuple[SHAPExplanation, dict]:    """Convenience: train XGBoost + compute SHAP in one call.    Args:        case_series: Daily case count array.        dates: Optional ISO date strings.        n_lags: Number of lag features.        top_k: Number of top features.    Returns:        Tuple of (SHAPExplanation, model_metrics_dict).    """    try:        import xgboost as xgb    except ImportError:        raise ImportError("XGBoost required for this function")    from .ml_forecaster import create_lag_features    df = create_lag_features(case_series, dates, n_lags=n_lags)    feature_cols = [        c for c in df.columns if c not in ("date", "target", "cases")    ]    X = df[feature_cols].values    y = df["target"].values    # Train    model = xgb.XGBRegressor(        n_estimators=200, max_depth=6, learning_rate=0.1,        random_state=42, verbosity=0,    )    model.fit(X, y, verbose=False)    # SHAP    explanation = explain_forecast(model, X, feature_cols, top_k=top_k)    metrics = {        "n_samples": len(X),        "n_features": len(feature_cols),        "model": "XGBoost",    }    return explanation, metrics

### Data Validator`epiagent/validators/epi_validator.py`

In [ ]:
"""Epidemiological Data Validator.Validates surveillance data for epidemiological plausibility BEFORE it reachesthe analysis engines. This is the critical "data quality firewall" that preventsgarbage-in-gospel-out failures.Validation Checks:    1. Non-negativity: cases >= 0, deaths >= 0    2. Biological plausibility: deaths <= cases (CFR ∈ [0, 1])    3. Population denominator: population > 0    4. Temporal spike detection: week-over-week change > 300% flagged    5. Completeness scoring: > 20% missing data degrades confidence    6. Date consistency: monotonically increasing, no future dates    7. Logical consistency: cumulative >= new (per day)The validator produces a DataQualityReport with a composite quality scoreQ ∈ [0, 1] that propagates through the pipeline to modulate confidencein downstream outputs."""from __future__ import annotationsimport loggingfrom dataclasses import dataclass, fieldfrom datetime import datetime, datefrom typing import Anyimport numpy as nplogger = logging.getLogger(__name__)@dataclassclass ValidationIssue:    """A single data quality issue found during validation."""    severity: str  # 'error', 'warning', 'info'    field: str    record_index: int    message: str    value: Any = None@dataclassclass DataQualityReport:    """Results of epidemiological data validation.    Attributes:        quality_score: Composite quality score in [0, 1]. Higher is better.        total_records: Number of records examined.        valid_records: Number of records passing all checks.        issues: List of all validation issues found.        is_acceptable: True if quality_score >= 0.7 (configurable threshold).    """    quality_score: float    total_records: int    valid_records: int    issues: list[ValidationIssue] = field(default_factory=list)    is_acceptable: bool = True    @property    def error_count(self) -> int:        return sum(1 for i in self.issues if i.severity == "error")    @property    def warning_count(self) -> int:        return sum(1 for i in self.issues if i.severity == "warning")    def summary(self) -> dict:        """Return a concise summary dict for agent state."""        return {            "quality_score": round(self.quality_score, 3),            "total_records": self.total_records,            "valid_records": self.valid_records,            "errors": self.error_count,            "warnings": self.warning_count,            "is_acceptable": self.is_acceptable,        }# ---------------------------------------------------------------------------# Spike detection threshold# ---------------------------------------------------------------------------_DEFAULT_SPIKE_THRESHOLD = 3.0       # 300% week-over-week increase_DEFAULT_COMPLETENESS_THRESHOLD = 0.8  # At least 80% non-missing records_DEFAULT_QUALITY_THRESHOLD = 0.7       # Minimum acceptable quality scoredef validate_surveillance_data(    records: list[dict],    *,    spike_threshold: float = _DEFAULT_SPIKE_THRESHOLD,    completeness_threshold: float = _DEFAULT_COMPLETENESS_THRESHOLD,    quality_threshold: float = _DEFAULT_QUALITY_THRESHOLD,) -> DataQualityReport:    """Validate a list of surveillance records for epidemiological plausibility.    Args:        records: List of surveillance record dicts. Expected keys:            date, region, pathogen, new_cases, cumulative_cases,            new_deaths, cumulative_deaths, population, source.        spike_threshold: Maximum acceptable fold-change in new_cases            between consecutive records (default 3.0 = 300%).        completeness_threshold: Minimum fraction of non-missing records            required (default 0.8 = 80%).        quality_threshold: Minimum quality_score for is_acceptable (default 0.7).    Returns:        DataQualityReport with composite quality score and all issues found.    """    if not records:        return DataQualityReport(            quality_score=0.0,            total_records=0,            valid_records=0,            issues=[ValidationIssue("error", "records", -1, "Empty dataset")],            is_acceptable=False,        )    issues: list[ValidationIssue] = []    total = len(records)    record_valid = [True] * total  # Track per-record validity    # -----------------------------------------------------------------------    # Pass 1: Per-record checks    # -----------------------------------------------------------------------    for idx, rec in enumerate(records):        # 1. Non-negativity check        for fld in ("new_cases", "cumulative_cases", "new_deaths", "cumulative_deaths"):            val = rec.get(fld)            if val is not None and val < 0:                issues.append(ValidationIssue(                    "error", fld, idx,                    f"Negative value: {fld}={val}. Epidemiological counts cannot be negative.",                    value=val,                ))                record_valid[idx] = False        # 2. Biological plausibility: deaths cannot exceed cases        new_cases = rec.get("new_cases", 0) or 0        new_deaths = rec.get("new_deaths", 0) or 0        if new_deaths > new_cases and new_cases > 0:            issues.append(ValidationIssue(                "warning", "new_deaths", idx,                f"Deaths ({new_deaths}) exceed cases ({new_cases}). "                f"Implied CFR={new_deaths / new_cases:.1%} > 100%. "                "Possible reporting lag or data error.",                value={"new_deaths": new_deaths, "new_cases": new_cases},            ))        cum_cases = rec.get("cumulative_cases", 0) or 0        cum_deaths = rec.get("cumulative_deaths", 0) or 0        if cum_deaths > cum_cases and cum_cases > 0:            issues.append(ValidationIssue(                "warning", "cumulative_deaths", idx,                f"Cumulative deaths ({cum_deaths}) exceed cumulative cases ({cum_cases}).",                value={"cumulative_deaths": cum_deaths, "cumulative_cases": cum_cases},            ))        # 3. Population denominator guard        population = rec.get("population")        if population is None or population <= 0:            issues.append(ValidationIssue(                "error", "population", idx,                f"Invalid population={population}. Must be > 0 to prevent "                "division-by-zero in incidence calculations.",                value=population,            ))            record_valid[idx] = False        # 4. Logical consistency: cumulative >= new        if cum_cases < new_cases:            issues.append(ValidationIssue(                "warning", "cumulative_cases", idx,                f"Cumulative cases ({cum_cases}) less than new cases ({new_cases}).",                value={"cumulative_cases": cum_cases, "new_cases": new_cases},            ))        # 5. Date validation        date_str = rec.get("date")        if date_str:            try:                rec_date = datetime.fromisoformat(date_str).date()                if rec_date > date.today():                    issues.append(ValidationIssue(                        "warning", "date", idx,                        f"Future date detected: {date_str}. "                        "Surveillance data should not contain future dates.",                        value=date_str,                    ))            except (ValueError, TypeError):                issues.append(ValidationIssue(                    "error", "date", idx,                    f"Invalid date format: {date_str}. Expected ISO 8601.",                    value=date_str,                ))                record_valid[idx] = False        else:            issues.append(ValidationIssue(                "error", "date", idx,                "Missing date field.",            ))            record_valid[idx] = False    # -----------------------------------------------------------------------    # Pass 2: Temporal checks (across records)    # -----------------------------------------------------------------------    case_series = []    for rec in records:        val = rec.get("new_cases")        case_series.append(val if val is not None else np.nan)    case_arr = np.array(case_series, dtype=float)    # 6. Temporal spike detection    for i in range(1, len(case_arr)):        prev = case_arr[i - 1]        curr = case_arr[i]        if np.isnan(prev) or np.isnan(curr) or prev <= 0:            continue        fold_change = curr / prev        if fold_change > spike_threshold:            issues.append(ValidationIssue(                "warning", "new_cases", i,                f"Temporal spike: {fold_change:.1f}x increase from day {i-1} "                f"({prev:.0f}) to day {i} ({curr:.0f}). "                f"Exceeds {spike_threshold:.0f}x threshold. "                "May be a reporting artifact or data dump.",                value={"fold_change": round(fold_change, 2)},            ))    # 7. Monotonicity check on cumulative fields    for fld in ("cumulative_cases", "cumulative_deaths"):        prev_val = None        for idx, rec in enumerate(records):            val = rec.get(fld)            if val is not None and prev_val is not None:                if val < prev_val:                    issues.append(ValidationIssue(                        "warning", fld, idx,                        f"Non-monotonic {fld}: decreased from {prev_val} to {val}. "                        "Cumulative counts should never decrease (possible data correction).",                        value={"previous": prev_val, "current": val},                    ))            prev_val = val    # 8. Date ordering    dates_parsed = []    for idx, rec in enumerate(records):        try:            dates_parsed.append(datetime.fromisoformat(rec.get("date", "")).date())        except (ValueError, TypeError):            dates_parsed.append(None)    for i in range(1, len(dates_parsed)):        if dates_parsed[i] is not None and dates_parsed[i - 1] is not None:            if dates_parsed[i] < dates_parsed[i - 1]:                issues.append(ValidationIssue(                    "warning", "date", i,                    f"Non-chronological dates: {dates_parsed[i]} follows {dates_parsed[i-1]}.",                ))    # -----------------------------------------------------------------------    # Pass 3: Completeness scoring    # -----------------------------------------------------------------------    required_fields = [        "date", "region", "pathogen", "new_cases", "population"    ]    missing_count = 0    total_field_checks = total * len(required_fields)    for idx, rec in enumerate(records):        for fld in required_fields:            if rec.get(fld) is None:                missing_count += 1    completeness = 1.0 - (missing_count / total_field_checks) if total_field_checks > 0 else 0.0    if completeness < completeness_threshold:        issues.append(ValidationIssue(            "warning", "completeness", -1,            f"Data completeness ({completeness:.1%}) below threshold "            f"({completeness_threshold:.1%}). Missing {missing_count} values "            f"across {total_field_checks} field checks.",            value={"completeness": round(completeness, 3)},        ))    # -----------------------------------------------------------------------    # Compute composite quality score    # -----------------------------------------------------------------------    valid_count = sum(record_valid)    error_count = sum(1 for i in issues if i.severity == "error")    warning_count = sum(1 for i in issues if i.severity == "warning")    # Quality score components:    #   - Record validity ratio (weight: 0.5)    #   - Completeness ratio (weight: 0.3)    #   - Warning penalty (weight: 0.2) — each warning reduces by 0.02, capped    validity_ratio = valid_count / total if total > 0 else 0.0    warning_penalty = min(warning_count * 0.02, 0.2)    quality_score = (        0.5 * validity_ratio        + 0.3 * completeness        + 0.2 * (1.0 - warning_penalty)    )    quality_score = max(0.0, min(1.0, quality_score))    # If any errors exist, cap quality at 0.5    if error_count > 0:        quality_score = min(quality_score, 0.5)    report = DataQualityReport(        quality_score=quality_score,        total_records=total,        valid_records=valid_count,        issues=issues,        is_acceptable=quality_score >= quality_threshold,    )    logger.info(        "Validation complete: score=%.3f, records=%d/%d valid, "        "errors=%d, warnings=%d, acceptable=%s",        quality_score, valid_count, total, error_count, warning_count,        report.is_acceptable,    )    return reportdef filter_valid_records(    records: list[dict],    report: DataQualityReport,) -> list[dict]:    """Return only records that passed all error-level checks.    Args:        records: Original list of surveillance record dicts.        report: DataQualityReport from validate_surveillance_data().    Returns:        Filtered list of records with no error-level issues.    """    error_indices = {        issue.record_index        for issue in report.issues        if issue.severity == "error" and issue.record_index >= 0    }    return [        rec for idx, rec in enumerate(records)        if idx not in error_indices    ]

### Security Guardrails`epiagent/guardrails/security.py`

In [ ]:
"""HIPAA-Aligned Security Guardrails for Epidemic Surveillance.Implements the Safe Harbor de-identification method per 45 CFR § 164.514(b)(2),which requires removal of 18 types of identifiers.This module provides:    1. PII Detection — regex-based scanning for all 18 HIPAA identifier types    2. PII Stripping — aggressive removal/redaction of detected PII    3. Schema Validation — enforcement for surveillance data structure    4. Data Provenance — SHA-256 hashing for audit trails    5. Output Sanitization — ensures no PII leaks into generated reportsReferences:    45 CFR § 164.514(b)(2) — Safe Harbor Method of De-identification    https://www.hhs.gov/hipaa/for-professionals/privacy/special-topics/de-identification/"""from __future__ import annotationsimport hashlibimport jsonimport loggingimport refrom dataclasses import dataclass, fieldlogger = logging.getLogger(__name__)@dataclassclass PIIPattern:    """A regex pattern for detecting a specific type of PII."""    name: str    pattern: str    replacement: str    compiled: re.Pattern = field(init=False, repr=False)    def __post_init__(self):        self.compiled = re.compile(self.pattern, re.IGNORECASE)# ---------------------------------------------------------------------------# HIPAA Safe Harbor: 18 Identifier Types# ---------------------------------------------------------------------------HIPAA_PATTERNS: list[PIIPattern] = [    # 1. Names (common patterns: First Last, Last, First)    PIIPattern(        name="name",        pattern=r"\b(?:Mr|Mrs|Ms|Dr|Prof)\.?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+\b",        replacement="[REDACTED_NAME]",    ),    # 2. Geographic data (street addresses, zip codes)    PIIPattern(        name="address",        pattern=r"\b\d{1,5}\s+(?:[A-Za-z]+\s+){1,3}(?:St|Street|Ave|Avenue|Blvd|Boulevard|Dr|Drive|Rd|Road|Ln|Lane|Way|Ct|Court)\b",        replacement="[REDACTED_ADDRESS]",    ),    PIIPattern(        name="zip_code",        pattern=r"\b\d{5}(?:-\d{4})?\b",        replacement="[REDACTED_ZIP]",    ),    # 3. Dates (MM/DD/YYYY, MM-DD-YYYY, Month DD, YYYY) — except year alone    PIIPattern(        name="date_mmddyyyy",        pattern=r"\b(?:0[1-9]|1[0-2])[/\-](?:0[1-9]|[12]\d|3[01])[/\-](?:19|20)\d{2}\b",        replacement="[REDACTED_DATE]",    ),    PIIPattern(        name="date_text",        pattern=r"\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4}\b",        replacement="[REDACTED_DATE]",    ),    # 4. Phone numbers (US formats)    PIIPattern(        name="phone",        pattern=r"\b(?:\+?1[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)?\d{3}[-.\s]?\d{4}\b",        replacement="[REDACTED_PHONE]",    ),    # 5. Fax numbers (same format, usually labeled)    PIIPattern(        name="fax",        pattern=r"(?:fax|facsimile)[:\s]*(?:\+?1[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)?\d{3}[-.\s]?\d{4}",        replacement="[REDACTED_FAX]",    ),    # 6. Email addresses    PIIPattern(        name="email",        pattern=r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b",        replacement="[REDACTED_EMAIL]",    ),    # 7. Social Security Numbers    PIIPattern(        name="ssn",        pattern=r"\b\d{3}[-\s]?\d{2}[-\s]?\d{4}\b",        replacement="[REDACTED_SSN]",    ),    # 8. Medical record numbers (common patterns: MRN-XXXXXXX)    PIIPattern(        name="mrn",        pattern=r"\b(?:MRN|Medical\s*Record\s*(?:Number|No|#))[:\s]*[A-Z0-9\-]{5,15}\b",        replacement="[REDACTED_MRN]",    ),    # 9. Health plan beneficiary numbers    PIIPattern(        name="health_plan_id",        pattern=r"\b(?:Health\s*Plan|Beneficiary|Member)\s*(?:ID|Number|No|#)[:\s]*[A-Z0-9\-]{5,20}\b",        replacement="[REDACTED_HEALTH_PLAN_ID]",    ),    # 10. Account numbers    PIIPattern(        name="account_number",        pattern=r"\b(?:Account|Acct)\s*(?:Number|No|#)[:\s]*\d{6,20}\b",        replacement="[REDACTED_ACCOUNT]",    ),    # 11. Certificate/license numbers    PIIPattern(        name="license",        pattern=r"\b(?:License|Certificate|Cert)\s*(?:Number|No|#)[:\s]*[A-Z0-9\-]{5,20}\b",        replacement="[REDACTED_LICENSE]",    ),    # 12. Vehicle identifiers (VIN)    PIIPattern(        name="vin",        pattern=r"\b[A-HJ-NPR-Z0-9]{17}\b",        replacement="[REDACTED_VIN]",    ),    # 13. Device identifiers / serial numbers    PIIPattern(        name="device_serial",        pattern=r"\b(?:Serial|Device)\s*(?:Number|No|#|ID)[:\s]*[A-Z0-9\-]{5,25}\b",        replacement="[REDACTED_DEVICE_ID]",    ),    # 14. Web URLs    PIIPattern(        name="url",        pattern=r"https?://[^\s\"'<>]+",        replacement="[REDACTED_URL]",    ),    # 15. IP addresses (IPv4)    PIIPattern(        name="ip_address",        pattern=r"\b(?:\d{1,3}\.){3}\d{1,3}\b",        replacement="[REDACTED_IP]",    ),    # 16. Biometric identifiers (text references)    PIIPattern(        name="biometric",        pattern=r"\b(?:fingerprint|retinal?\s*scan|voice\s*print|facial\s*recognition|dna\s*sample)\b",        replacement="[REDACTED_BIOMETRIC]",    ),    # 17. Full face photographs (filename references)    PIIPattern(        name="photo_file",        pattern=r"\b\w+(?:_photo|_face|_headshot|_portrait)\.\w{3,4}\b",        replacement="[REDACTED_PHOTO]",    ),    # 18. Any other unique identifying number (catch-all for labeled IDs)    PIIPattern(        name="unique_id",        pattern=r"\b(?:Patient|Subject|Participant)\s*(?:ID|Number|No|#)[:\s]*[A-Z0-9\-]{3,20}\b",        replacement="[REDACTED_ID]",    ),]@dataclassclass SecurityReport:    """Results of security audit on surveillance data."""    pii_detected: bool    pii_types_found: list[str] = field(default_factory=list)    items_redacted: int = 0    data_hash: str = ""    schema_valid: bool = True    schema_errors: list[str] = field(default_factory=list)    def summary(self) -> dict:        """Return concise summary dict for agent state."""        return {            "pii_detected": self.pii_detected,            "pii_types": self.pii_types_found,            "items_redacted": self.items_redacted,            "data_hash": self.data_hash[:16] + "..." if self.data_hash else "",            "schema_valid": self.schema_valid,            "schema_errors_count": len(self.schema_errors),        }# ---------------------------------------------------------------------------# Core Functions# ---------------------------------------------------------------------------def detect_pii(text: str) -> list[tuple[str, str, int, int]]:    """Scan text for PII patterns.    Args:        text: Text to scan.    Returns:        List of (pattern_name, matched_text, start_pos, end_pos) tuples.    """    findings = []    for pattern in HIPAA_PATTERNS:        for match in pattern.compiled.finditer(text):            findings.append((                pattern.name,                match.group(),                match.start(),                match.end(),            ))    return findingsdef strip_pii(text: str) -> tuple[str, list[str]]:    """Remove all detected PII from text.    Args:        text: Text to sanitize.    Returns:        Tuple of (cleaned_text, list_of_redacted_pattern_names).    """    redacted_types = []    cleaned = text    for pattern in HIPAA_PATTERNS:        matches = pattern.compiled.findall(cleaned)        if matches:            cleaned = pattern.compiled.sub(pattern.replacement, cleaned)            redacted_types.append(pattern.name)    return cleaned, redacted_typesdef strip_pii_from_records(    records: list[dict],) -> tuple[list[dict], SecurityReport]:    """Strip PII from all string fields in surveillance records.    Args:        records: List of surveillance record dicts.    Returns:        Tuple of (cleaned_records, SecurityReport).    """    cleaned_records = []    all_redacted_types = set()    total_redacted = 0    for rec in records:        cleaned_rec = {}        for key, value in rec.items():            if isinstance(value, str):                cleaned_value, types = strip_pii(value)                if types:                    all_redacted_types.update(types)                    total_redacted += len(types)                cleaned_rec[key] = cleaned_value            elif isinstance(value, dict):                # Recurse into nested dicts                inner_cleaned = {}                for k, v in value.items():                    if isinstance(v, str):                        cv, t = strip_pii(v)                        if t:                            all_redacted_types.update(t)                            total_redacted += len(t)                        inner_cleaned[k] = cv                    else:                        inner_cleaned[k] = v                cleaned_rec[key] = inner_cleaned            else:                cleaned_rec[key] = value        cleaned_records.append(cleaned_rec)    pii_detected = total_redacted > 0    report = SecurityReport(        pii_detected=pii_detected,        pii_types_found=sorted(all_redacted_types),        items_redacted=total_redacted,        data_hash=compute_data_hash(records),        schema_valid=True,  # Schema validation done separately    )    if pii_detected:        logger.warning(            "PII detected and stripped: %d items, types: %s",            total_redacted, sorted(all_redacted_types),        )    else:        logger.info("No PII detected in %d records.", len(records))    return cleaned_records, reportdef validate_schema(data: dict | list) -> tuple[bool, list[str]]:    """Validate surveillance data structure.    Checks that required fields exist with correct types.    Args:        data: Single record dict or list of record dicts.    Returns:        Tuple of (is_valid, list_of_error_messages).    """    required_fields = {        "date": str,        "region": str,        "pathogen": str,        "new_cases": (int, float),        "population": (int, float),    }    optional_fields = {        "cumulative_cases": (int, float),        "new_deaths": (int, float),        "cumulative_deaths": (int, float),        "source": str,        "metadata": dict,    }    errors = []    records = data if isinstance(data, list) else [data]    for idx, rec in enumerate(records):        if not isinstance(rec, dict):            errors.append(f"Record {idx}: expected dict, got {type(rec).__name__}")            continue        for field_name, expected_type in required_fields.items():            if field_name not in rec:                errors.append(f"Record {idx}: missing required field '{field_name}'")            elif not isinstance(rec[field_name], expected_type):                errors.append(                    f"Record {idx}: field '{field_name}' expected "                    f"{expected_type}, got {type(rec[field_name]).__name__}"                )    is_valid = len(errors) == 0    return is_valid, errorsdef compute_data_hash(data: object) -> str:    """Compute SHA-256 hash of data for provenance tracking.    Args:        data: Any JSON-serializable object.    Returns:        Hex-encoded SHA-256 hash string.    """    serialized = json.dumps(data, sort_keys=True, default=str)    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()def create_security_report(records: list[dict]) -> SecurityReport:    """Run full security audit on a dataset.    Performs PII detection, schema validation, and data hashing.    Args:        records: List of surveillance record dicts.    Returns:        Complete SecurityReport.    """    # Schema validation    schema_valid, schema_errors = validate_schema(records)    # PII scan (without modifying data)    all_pii_types = set()    total_pii = 0    for rec in records:        for key, value in rec.items():            if isinstance(value, str):                findings = detect_pii(value)                if findings:                    total_pii += len(findings)                    all_pii_types.update(f[0] for f in findings)    return SecurityReport(        pii_detected=total_pii > 0,        pii_types_found=sorted(all_pii_types),        items_redacted=0,  # This is audit-only, no redaction performed        data_hash=compute_data_hash(records),        schema_valid=schema_valid,        schema_errors=schema_errors,    )

### Data Schemas`epiagent/schemas/surveillance.py`

In [ ]:
"""Pydantic Data Schemas for Epidemic Surveillance.Defines the canonical data models used throughout the EpiAgent pipeline.All inter-agent communication uses these schemas for type safety and validation."""from __future__ import annotationsimport uuidfrom datetime import datetimefrom enum import Enumfrom typing import Any, Literalfrom pydantic import BaseModel, Field# ---------------------------------------------------------------------------# Core data models# ---------------------------------------------------------------------------class SurveillanceRecord(BaseModel):    """Single day/week surveillance observation."""    date: str = Field(..., description="ISO 8601 date string (YYYY-MM-DD)")    region: str = Field(..., description="Geographic region identifier")    pathogen: str = Field(..., description="Pathogen name (e.g., 'influenza', 'covid-19')")    new_cases: int = Field(..., ge=0, description="New cases reported this period")    cumulative_cases: int = Field(..., ge=0, description="Total cumulative cases")    new_deaths: int = Field(0, ge=0, description="New deaths reported this period")    cumulative_deaths: int = Field(0, ge=0, description="Total cumulative deaths")    population: int = Field(..., gt=0, description="Population at risk")    source: str = Field(..., description="Data source identifier")    metadata: dict[str, Any] = Field(default_factory=dict)class SurveillanceDataset(BaseModel):    """Collection of surveillance records with provenance metadata."""    records: list[SurveillanceRecord]    fetch_timestamp: str = Field(        default_factory=lambda: datetime.utcnow().isoformat(),        description="ISO 8601 datetime when data was fetched",    )    source_hash: str = Field("", description="SHA-256 hash of raw data for audit trail")    @property    def record_count(self) -> int:        return len(self.records)    @property    def date_range(self) -> tuple[str, str] | None:        if not self.records:            return None        dates = sorted(r.date for r in self.records)        return (dates[0], dates[-1])    @property    def regions(self) -> list[str]:        return sorted(set(r.region for r in self.records))    @property    def pathogens(self) -> list[str]:        return sorted(set(r.pathogen for r in self.records))# ---------------------------------------------------------------------------# Data quality models# ---------------------------------------------------------------------------class DataQualityIssue(BaseModel):    """Individual validation issue found during data quality checks."""    severity: Literal["error", "warning", "info"]    field: str    record_index: int    message: str    value: Any = Noneclass DataQualityReport(BaseModel):    """Output of epidemiological data validation."""    quality_score: float = Field(..., ge=0.0, le=1.0)    total_records: int    valid_records: int    issues: list[DataQualityIssue] = Field(default_factory=list)    @property    def is_acceptable(self) -> bool:        return self.quality_score >= 0.7# ---------------------------------------------------------------------------# Metric models# ---------------------------------------------------------------------------class MetricWithCI(BaseModel):    """Single metric with confidence interval."""    value: float    lower_ci: float    upper_ci: float    method: strclass EpiMetrics(BaseModel):    """Bundle of epidemiological metrics."""    cfr: MetricWithCI    incidence_rate: MetricWithCI    doubling_time: MetricWithCI | None = None    attack_rate: MetricWithCI    growth_rate: MetricWithCI | None = None# ---------------------------------------------------------------------------# Rt estimation models# ---------------------------------------------------------------------------class RtEstimate(BaseModel):    """Rt estimation result for serialization."""    dates: list[str]    rt_mean: list[float]    rt_lower: list[float]    rt_upper: list[float]    current_rt: float    current_phase: Literal["growing", "declining", "stable", "unknown"]# ---------------------------------------------------------------------------# Forecast models# ---------------------------------------------------------------------------class ForecastResult(BaseModel):    """ML forecast output."""    dates: list[str]    predicted: list[float]    lower_bound: list[float]    upper_bound: list[float]    model_name: str    rmse: float | None = None    feature_importance: dict[str, float] = Field(default_factory=dict)# ---------------------------------------------------------------------------# Alert models# ---------------------------------------------------------------------------class AlertLevel(str, Enum):    """Public health alert level classification."""    GREEN = "GREEN"     # Normal activity, Rt < 0.8    YELLOW = "YELLOW"   # Elevated activity, 0.8 <= Rt < 1.0    ORANGE = "ORANGE"   # High activity, 1.0 <= Rt < 1.5    RED = "RED"         # Critical activity, Rt >= 1.5 or rapid growth# ---------------------------------------------------------------------------# Situation Report# ---------------------------------------------------------------------------class SituationReport(BaseModel):    """Executive Situation Report (SitRep) — the final pipeline output."""    report_id: str = Field(        default_factory=lambda: str(uuid.uuid4()),        description="Unique report identifier",    )    generated_at: str = Field(        default_factory=lambda: datetime.utcnow().isoformat(),        description="ISO 8601 datetime of report generation",    )    region: str    pathogen: str    alert_level: AlertLevel    executive_summary: str = Field(        ..., description="1-paragraph executive summary for decision makers"    )    epi_metrics: EpiMetrics    rt_estimate: RtEstimate    forecasts: list[ForecastResult] = Field(default_factory=list)    data_quality: DataQualityReport    recommendations: list[str] = Field(default_factory=list)    methodology_notes: str = Field(        "",        description="Technical methodology description for reproducibility",    )    reproducibility: dict[str, Any] = Field(        default_factory=dict,        description="Software versions, data hash, parameters for full reproducibility",    )

### Synthetic Data Generator`mcp_server/data_sources/synthetic.py`

In [ ]:
"""Synthetic Epidemic Data Generator.Generates realistic synthetic surveillance data using SEIR dynamicswith configurable noise for three pathogen profiles:    - Influenza-like: R0≈1.3, serial interval 2.6 days, CFR≈0.1%    - COVID-like: R0≈2.5, serial interval 4.7 days, CFR≈1.5%    - Measles-like: R0≈12, serial interval 11.5 days, CFR≈0.2%The generator produces daily surveillance records with:    - Realistic case counts from SEIR dynamics + Poisson noise    - Deaths derived from cases with appropriate CFR + binomial noise    - Configurable reporting delays and weekend effects"""from __future__ import annotationsimport loggingfrom dataclasses import dataclassfrom datetime import date, timedeltaimport numpy as npfrom scipy.integrate import solve_ivplogger = logging.getLogger(__name__)@dataclassclass PathogenProfile:    """Configuration for a pathogen's epidemiological characteristics."""    name: str    R0: float    latent_period: float      # days    infectious_period: float  # days    cfr: float                # case fatality rate (0-1)    population: int = 1_000_000# ---------------------------------------------------------------------------# Preset pathogen profiles# ---------------------------------------------------------------------------INFLUENZA = PathogenProfile(    name="influenza",    R0=1.3,    latent_period=2.0,    infectious_period=3.0,    cfr=0.001,    population=1_000_000,)COVID = PathogenProfile(    name="covid-19",    R0=2.5,    latent_period=5.2,    infectious_period=2.9,    cfr=0.015,    population=1_000_000,)MEASLES = PathogenProfile(    name="measles",    R0=12.0,    latent_period=10.0,    infectious_period=8.0,    cfr=0.002,    population=1_000_000,)@dataclassclass SurveillanceRecord:    """A single daily surveillance observation."""    date: str    region: str    pathogen: str    new_cases: int    cumulative_cases: int    new_deaths: int    cumulative_deaths: int    population: int    source: str = "synthetic"    def to_dict(self) -> dict:        """Convert to dictionary for JSON serialization."""        return {            "date": self.date,            "region": self.region,            "pathogen": self.pathogen,            "new_cases": self.new_cases,            "cumulative_cases": self.cumulative_cases,            "new_deaths": self.new_deaths,            "cumulative_deaths": self.cumulative_deaths,            "population": self.population,            "source": self.source,            "metadata": {},        }def _run_seir_internal(    N: int,    beta: float,    sigma: float,    gamma: float,    I0: float,    E0: float,    t_max: int,) -> tuple[np.ndarray, np.ndarray]:    """Run SEIR model and return daily incidence.    Self-contained ODE solver (doesn't import from engines to keep    MCP server independent for deployment).    Returns:        Tuple of (time_array, daily_incidence_array).    """    S0 = N - E0 - I0    def odes(t, y):        S, E, I, R = y        dS = -beta * S * I / N        dE = beta * S * I / N - sigma * E        dI = sigma * E - gamma * I        dR = gamma * I        return [dS, dE, dI, dR]    t_eval = np.arange(t_max + 1, dtype=float)    sol = solve_ivp(        odes,        t_span=(0, t_max),        y0=[S0, E0, I0, 0.0],        t_eval=t_eval,        method="RK45",        rtol=1e-8,        atol=1e-8,    )    S = sol.y[0]    I = sol.y[2]    # Daily incidence = β·S·I/N    incidence = beta * S * I / N    return sol.t, np.maximum(incidence, 0.0)def generate_epidemic(    profile: PathogenProfile,    duration_days: int = 180,    initial_infected: int = 10,    noise_level: float = 0.1,    seed: int = 42,    region: str = "synthetic_region",    start_date: str | None = None,) -> list[SurveillanceRecord]:    """Generate synthetic epidemic surveillance data.    Args:        profile: PathogenProfile defining the disease characteristics.        duration_days: Number of days to simulate.        initial_infected: Number of initially infectious individuals.        noise_level: Poisson noise multiplier (0 = deterministic).        seed: Random seed for reproducibility.        region: Region name for records.        start_date: ISO date string for day 0 (defaults to today - duration).    Returns:        List of SurveillanceRecord objects.    """    rng = np.random.default_rng(seed)    # SEIR parameters    gamma = 1.0 / profile.infectious_period    sigma = 1.0 / profile.latent_period    beta = profile.R0 * gamma    # Run deterministic SEIR    t, true_incidence = _run_seir_internal(        N=profile.population,        beta=beta,        sigma=sigma,        gamma=gamma,        I0=float(initial_infected),        E0=0.0,        t_max=duration_days,    )    # Start date    if start_date:        base_date = date.fromisoformat(start_date)    else:        base_date = date.today() - timedelta(days=duration_days)    # Generate noisy observations    records = []    cumulative_cases = 0    cumulative_deaths = 0    for day_idx in range(len(true_incidence)):        current_date = base_date + timedelta(days=day_idx)        # Add Poisson noise to incidence        expected = true_incidence[day_idx]        if noise_level > 0 and expected > 0:            noisy = rng.poisson(max(0, expected * (1.0 + noise_level * rng.standard_normal())))        else:            noisy = int(round(expected))        new_cases = max(0, int(noisy))        # Weekend reporting effect: reduce by 30%, redistribute to Monday        weekday = current_date.weekday()        if weekday in (5, 6):  # Saturday, Sunday            weekend_reduction = int(new_cases * 0.3)            new_cases -= weekend_reduction        elif weekday == 0:  # Monday            # Add back some weekend cases (approximation)            new_cases = int(new_cases * 1.4)        # Derive deaths from cases using binomial sampling        if new_cases > 0 and profile.cfr > 0:            new_deaths = int(rng.binomial(new_cases, profile.cfr))        else:            new_deaths = 0        cumulative_cases += new_cases        cumulative_deaths += new_deaths        record = SurveillanceRecord(            date=current_date.isoformat(),            region=region,            pathogen=profile.name,            new_cases=new_cases,            cumulative_cases=cumulative_cases,            new_deaths=new_deaths,            cumulative_deaths=cumulative_deaths,            population=profile.population,            source="synthetic",        )        records.append(record)    logger.info(        "Generated %d-day %s epidemic: total cases=%d, total deaths=%d",        duration_days, profile.name, cumulative_cases, cumulative_deaths,    )    return recordsdef generate_multi_wave(    profile: PathogenProfile,    num_waves: int = 2,    duration_days: int = 365,    wave_gap_days: int = 60,    r0_decay: float = 0.7,    seed: int = 42,    region: str = "synthetic_region",) -> list[SurveillanceRecord]:    """Generate multi-wave epidemic data.    Each subsequent wave has a reduced R0 (simulating immunity/vaccination).    Args:        profile: Base PathogenProfile.        num_waves: Number of epidemic waves.        duration_days: Total duration.        wave_gap_days: Gap between wave starts.        r0_decay: R0 multiplier for each subsequent wave.        seed: Random seed.        region: Region name.    Returns:        List of SurveillanceRecord objects.    """    rng = np.random.default_rng(seed)    base_date = date.today() - timedelta(days=duration_days)    # Generate each wave separately and combine    all_incidence = np.zeros(duration_days + 1)    for wave in range(num_waves):        wave_r0 = profile.R0 * (r0_decay ** wave)        gamma = 1.0 / profile.infectious_period        sigma = 1.0 / profile.latent_period        beta = wave_r0 * gamma        wave_start = wave * wave_gap_days        wave_duration = duration_days - wave_start        if wave_duration <= 0:            break        _, wave_incidence = _run_seir_internal(            N=profile.population,            beta=beta,            sigma=sigma,            gamma=gamma,            I0=10.0,            E0=0.0,            t_max=wave_duration,        )        # Add to total incidence        for i, inc in enumerate(wave_incidence):            if wave_start + i < len(all_incidence):                all_incidence[wave_start + i] += inc    # Generate records from combined incidence    records = []    cumulative_cases = 0    cumulative_deaths = 0    for day_idx in range(len(all_incidence)):        current_date = base_date + timedelta(days=day_idx)        expected = all_incidence[day_idx]        new_cases = max(0, int(rng.poisson(max(0, expected))))        if new_cases > 0 and profile.cfr > 0:            new_deaths = int(rng.binomial(new_cases, profile.cfr))        else:            new_deaths = 0        cumulative_cases += new_cases        cumulative_deaths += new_deaths        records.append(SurveillanceRecord(            date=current_date.isoformat(),            region=region,            pathogen=profile.name,            new_cases=new_cases,            cumulative_cases=cumulative_cases,            new_deaths=new_deaths,            cumulative_deaths=cumulative_deaths,            population=profile.population,            source="synthetic",        ))    logger.info(        "Generated %d-wave %s epidemic over %d days: "        "total cases=%d, total deaths=%d",        num_waves, profile.name, duration_days,        cumulative_cases, cumulative_deaths,    )    return records

### CDC FluView Client`mcp_server/data_sources/cdc_fluview.py`

In [ ]:
"""CDC FluView ILINet Data Client.Fetches Influenza-Like Illness (ILI) surveillance data from theCMU Delphi Epidata API — the standard programmatic interface forCDC ILINet data used by epidemiology researchers.API Docs: https://cmu-delphi.github.io/delphi-epidata/Endpoint: https://api.delphi.cmu.edu/epidata/fluview/Data Available:    - Weekly ILI percentages by HHS region and nationally    - Patient counts, provider counts    - Available from 1997 to present"""from __future__ import annotationsimport loggingimport timefrom datetime import date, timedeltaimport requestsfrom .synthetic import SurveillanceRecordlogger = logging.getLogger(__name__)BASE_URL = "https://api.delphi.cmu.edu/epidata"# Rate limiting: max 1 request per second_last_request_time = 0.0def _rate_limit():    """Enforce 1 request per second rate limiting."""    global _last_request_time    elapsed = time.time() - _last_request_time    if elapsed < 1.0:        time.sleep(1.0 - elapsed)    _last_request_time = time.time()def epiweek_to_date(epiweek: int) -> str:    """Convert CDC epiweek integer (YYYYWW) to ISO date string.    The CDC epiweek starts on Sunday. We return the date of the    Saturday (end of the epiweek) for consistency.    Args:        epiweek: Integer in YYYYWW format (e.g., 202301).    Returns:        ISO date string (YYYY-MM-DD).    """    year = epiweek // 100    week = epiweek % 100    # January 4th is always in week 1 (ISO standard)    jan4 = date(year, 1, 4)    # Find the Monday of ISO week 1    iso_week1_monday = jan4 - timedelta(days=jan4.weekday())    # CDC weeks start on Sunday, so subtract 1 day from Monday    cdc_week1_start = iso_week1_monday - timedelta(days=1)    # Target week start (Sunday)    target_start = cdc_week1_start + timedelta(weeks=week - 1)    # End of week (Saturday)    target_end = target_start + timedelta(days=6)    return target_end.isoformat()def fetch_fluview(    regions: str = "nat",    epiweeks: str = "202301-202352",) -> list[SurveillanceRecord]:    """Fetch ILINet data from CMU Delphi Epidata API.    Args:        regions: Comma-separated region codes.            'nat' = national, 'hhs1'-'hhs10' = HHS regions,            or 2-letter state codes.        epiweeks: Epiweek range in YYYYWW-YYYYWW format.    Returns:        List of SurveillanceRecord objects. Empty list on error.    """    _rate_limit()    url = f"{BASE_URL}/fluview/"    params = {        "regions": regions,        "epiweeks": epiweeks,    }    try:        response = requests.get(url, params=params, timeout=30)        response.raise_for_status()        data = response.json()        if data.get("result") != 1:            logger.warning(                "Delphi API returned non-success: %s",                data.get("message", "Unknown error"),            )            return []        records = []        epi_data = data.get("epidata", [])        for entry in epi_data:            # Extract relevant fields            epiweek = entry.get("epiweek", 0)            region = entry.get("region", "unknown")            num_ili = entry.get("num_ili", 0)            num_patients = entry.get("num_patients", 0)            # Convert epiweek to date            date_str = epiweek_to_date(epiweek)            # ILINet doesn't report deaths directly; set to 0            # num_ili = number of ILI cases seen by sentinel providers            record = SurveillanceRecord(                date=date_str,                region=region,                pathogen="influenza",                new_cases=int(num_ili) if num_ili else 0,                cumulative_cases=0,  # ILINet reports weekly, not cumulative                new_deaths=0,                cumulative_deaths=0,                population=330_000_000 if region == "nat" else 33_000_000,                source="cdc_fluview",            )            records.append(record)        logger.info(            "Fetched %d records from CDC FluView (regions=%s, epiweeks=%s)",            len(records), regions, epiweeks,        )        return records    except requests.RequestException as e:        logger.error("Failed to fetch CDC FluView data: %s", e)        return []    except (ValueError, KeyError) as e:        logger.error("Failed to parse CDC FluView response: %s", e)        return []def fetch_fluview_meta() -> dict:    """Fetch metadata about available FluView data.    Returns:        Dict with metadata or empty dict on error.    """    _rate_limit()    url = f"{BASE_URL}/fluview_meta/"    try:        response = requests.get(url, timeout=30)        response.raise_for_status()        data = response.json()        return data.get("epidata", {})    except requests.RequestException as e:        logger.error("Failed to fetch FluView metadata: %s", e)        return {}

### FunctionTool Wrappers`epiagent/agents/tools.py`

In [ ]:
"""FunctionTool wrappers for EpiAgent computational engines.Each function is designed to be invoked as a deterministic FunctionToolby an LLM agent. The LLM agent decides WHEN to call these tools andINTERPRETS the results — but the computation itself is fully deterministic.This is the critical architectural pattern:    LLM decides → FunctionTool computes → LLM interprets    (No hallucinated math. Ever.)"""from __future__ import annotationsimport jsonimport loggingimport tracebackimport numpy as nplogger = logging.getLogger(__name__)# ---------------------------------------------------------------------------# Data Retrieval Tools# ---------------------------------------------------------------------------def fetch_synthetic_data(    pathogen: str = "covid-19",    duration_days: int = 180,    region: str = "synthetic_region",    noise_level: float = 0.1,    seed: int = 42,) -> dict:    """Fetch synthetic epidemic surveillance data.    Generates realistic epidemic data using SEIR dynamics with    configurable Poisson noise for testing and demonstration.    Args:        pathogen: One of 'influenza', 'covid-19', 'measles'.        duration_days: Number of days to simulate.        region: Region name for records.        noise_level: Noise multiplier (0=deterministic, 0.1=default).        seed: Random seed for reproducibility.    Returns:        Dict with 'records' (list of surveillance dicts), 'summary' stats.    """    from mcp_server.data_sources.synthetic import (        INFLUENZA, COVID, MEASLES, generate_epidemic,    )    profiles = {"influenza": INFLUENZA, "covid-19": COVID, "measles": MEASLES}    profile = profiles.get(pathogen.lower())    if not profile:        return {"error": f"Unknown pathogen '{pathogen}'. Options: {list(profiles.keys())}"}    records = generate_epidemic(        profile=profile,        duration_days=duration_days,        noise_level=noise_level,        seed=seed,        region=region,    )    record_dicts = [r.to_dict() for r in records]    total_cases = sum(r.new_cases for r in records)    total_deaths = sum(r.new_deaths for r in records)    peak_day = max(range(len(records)), key=lambda i: records[i].new_cases)    return {        "record_count": len(record_dicts),        "pathogen": pathogen,        "region": region,        "date_range": f"{records[0].date} to {records[-1].date}",        "total_cases": total_cases,        "total_deaths": total_deaths,        "peak_day": records[peak_day].date,        "peak_cases": records[peak_day].new_cases,        "records": record_dicts,    }def fetch_cdc_data(    regions: str = "nat",    epiweeks: str = "202301-202352",) -> dict:    """Fetch real CDC FluView ILINet surveillance data.    Retrieves weekly ILI data from the CMU Delphi Epidata API.    Args:        regions: Region codes: 'nat' for national, 'hhs1'-'hhs10'.        epiweeks: Epiweek range (YYYYWW-YYYYWW).    Returns:        Dict with 'records' and summary stats.    """    from mcp_server.data_sources.cdc_fluview import fetch_fluview    records = fetch_fluview(regions=regions, epiweeks=epiweeks)    record_dicts = [r.to_dict() for r in records]    if not records:        return {"record_count": 0, "records": [], "warning": "No data returned from CDC API"}    return {        "record_count": len(record_dicts),        "source": "cdc_fluview",        "regions": regions,        "records": record_dicts,    }# ---------------------------------------------------------------------------# Validation Tools# ---------------------------------------------------------------------------def validate_data(records_json: str) -> dict:    """Run epidemiological data quality validation.    Applies 8-point plausibility checks on surveillance data.    Args:        records_json: JSON string of surveillance records list.    Returns:        Dict with quality_score, issues, and recommendations.    """    from epiagent.validators.epi_validator import validate_surveillance_data    try:        records = json.loads(records_json) if isinstance(records_json, str) else records_json    except json.JSONDecodeError as e:        return {"error": f"Invalid JSON: {e}"}    report = validate_surveillance_data(records)    return {        "quality_score": report.quality_score,        "is_acceptable": report.is_acceptable,        "total_records": report.total_records,        "valid_records": report.valid_records,        "error_count": report.error_count,        "warning_count": report.warning_count,        "issues": [            {                "severity": i.severity,                "field": i.field,                "record_index": i.record_index,                "message": i.message,            }            for i in report.issues[:20]  # Limit to 20 for LLM context        ],    }# ---------------------------------------------------------------------------# Security Tools# ---------------------------------------------------------------------------def run_security_audit(records_json: str) -> dict:    """Run HIPAA-aligned security audit on surveillance data.    Scans for all 18 HIPAA identifier types and validates schema.    Args:        records_json: JSON string of surveillance records list.    Returns:        Dict with PII findings, schema validation, and data hash.    """    from epiagent.guardrails.security import (        strip_pii_from_records, create_security_report,    )    try:        records = json.loads(records_json) if isinstance(records_json, str) else records_json    except json.JSONDecodeError as e:        return {"error": f"Invalid JSON: {e}"}    # Audit    report = create_security_report(records)    # Strip PII if found    cleaned_records = records    if report.pii_detected:        cleaned_records, strip_report = strip_pii_from_records(records)        report = strip_report    return {        "pii_detected": report.pii_detected,        "pii_types_found": report.pii_types_found,        "items_redacted": report.items_redacted,        "data_hash": report.data_hash,        "schema_valid": report.schema_valid,        "schema_errors": report.schema_errors[:10],        "cleaned_records": cleaned_records,    }# ---------------------------------------------------------------------------# Epidemiological Analysis Tools# ---------------------------------------------------------------------------def run_seir_model(    population: int = 1_000_000,    R0: float = 2.5,    latent_period: float = 5.2,    infectious_period: float = 2.9,    initial_infected: int = 10,    t_max: int = 365,) -> dict:    """Run SEIR compartmental model simulation.    Solves the SEIR differential equations using RK45.    Args:        population: Total population size.        R0: Basic reproduction number.        latent_period: Average latent period (days).        infectious_period: Average infectious period (days).        initial_infected: Initial number of infectious individuals.        t_max: Simulation duration (days).    Returns:        Dict with model outputs, peak timing, and attack rate.    """    from epiagent.engines.seir_model import SEIRParameters, run_seir    params = SEIRParameters.from_epi_params(        R0=R0,        latent_period=latent_period,        infectious_period=infectious_period,        population=population,        initial_infected=initial_infected,    )    result = run_seir(params, t_max=t_max)    return {        "summary": result.summary(),        "daily_incidence": result.daily_incidence[:60].tolist(),  # First 60 days        "peak_day": int(result.peak_day),        "peak_cases": float(result.peak_cases),        "total_infected": float(result.R[-1]),        "attack_rate": float(result.R[-1] / params.N),    }def estimate_rt(    incidence_json: str,    pathogen: str = "covid-19",    window: int = 7,) -> dict:    """Estimate time-varying effective reproduction number Rt.    Uses the Cori et al. (2013) Bayesian method — the WHO/CDC standard.    Args:        incidence_json: JSON array of daily case counts.        pathogen: Pathogen name to select serial interval.        window: Sliding window size in days.    Returns:        Dict with Rt estimates, credible intervals, and phase classification.    """    from epiagent.engines.rt_estimation import (        estimate_rt as _estimate_rt,        SI_COVID, SI_INFLUENZA, SI_MEASLES,    )    try:        incidence = np.array(json.loads(incidence_json), dtype=float)    except (json.JSONDecodeError, TypeError) as e:        return {"error": f"Invalid incidence data: {e}"}    si_map = {        "covid-19": SI_COVID, "covid": SI_COVID,        "influenza": SI_INFLUENZA, "flu": SI_INFLUENZA,        "measles": SI_MEASLES,    }    si = si_map.get(pathogen.lower(), SI_COVID)    try:        result = _estimate_rt(incidence, si, window=window)    except ValueError as e:        return {"error": str(e)}    # Return last 30 days of Rt    n = min(30, len(result.rt_mean))    return {        "current_rt": result.current_rt,        "current_phase": result.current_phase,        "rt_last_30d": {            "rt_mean": [round(x, 3) if not np.isnan(x) else None for x in result.rt_mean[-n:]],            "rt_lower": [round(x, 3) if not np.isnan(x) else None for x in result.rt_lower[-n:]],            "rt_upper": [round(x, 3) if not np.isnan(x) else None for x in result.rt_upper[-n:]],            "phase": result.epidemic_phase[-n:],        },        "summary": result.summary(),    }def compute_epi_metrics(    total_cases: int,    total_deaths: int,    population: int,    case_series_json: str = "[]",) -> dict:    """Compute standard epidemiological metrics with confidence intervals.    Computes: CFR, incidence rate, attack rate, doubling time, growth rate.    Args:        total_cases: Total cumulative cases.        total_deaths: Total cumulative deaths.        population: Population at risk.        case_series_json: JSON array of daily case counts (for time-based metrics).    Returns:        Dict with all metrics and their confidence intervals.    """    from epiagent.engines.epi_metrics import (        compute_cfr, compute_incidence_rate, compute_attack_rate,        compute_doubling_time, compute_growth_rate,    )    results = {}    # CFR    cfr = compute_cfr(total_deaths, total_cases)    results["cfr"] = cfr.summary()    # Incidence rate    inc = compute_incidence_rate(total_cases, population)    results["incidence_rate_per_100k"] = inc.summary()    # Attack rate    ar = compute_attack_rate(total_cases, population)    results["attack_rate"] = ar.summary()    # Time-based metrics (if series provided)    try:        case_series = np.array(json.loads(case_series_json), dtype=float)        if len(case_series) >= 7:            dt = compute_doubling_time(case_series)            results["doubling_time_days"] = dt.summary()            gr = compute_growth_rate(case_series)            results["growth_rate_per_day"] = gr.summary()    except (json.JSONDecodeError, TypeError):        pass    return resultsdef detect_changepoints(    case_series_json: str,    hazard_lambda: float = 100.0,) -> dict:    """Detect outbreak changepoints using Bayesian Online Changepoint Detection.    Identifies abrupt changes in epidemic dynamics (e.g., outbreak onset,    lockdown effects, new variant emergence).    Args:        case_series_json: JSON array of daily case counts.        hazard_lambda: Expected run length (higher = fewer changepoints).    Returns:        Dict with detected changepoints and their confidence scores.    """    from epiagent.engines.changepoint_detector import (        detect_outbreak_signals as _detect_cps,    )    try:        case_series = np.array(json.loads(case_series_json), dtype=float)    except (json.JSONDecodeError, TypeError) as e:        return {"error": f"Invalid data: {e}"}    try:        result = _detect_cps(case_series, hazard_lambda=hazard_lambda)        # Extract confidence scores from changepoint_probs at detected indices        confidence_scores = [            float(result.changepoint_probs[cp])            for cp in result.changepoints            if cp < len(result.changepoint_probs)        ]        return {            "changepoints": result.changepoints,            "confidence_scores": [round(s, 3) for s in confidence_scores],            "n_changepoints": len(result.changepoints),            "summary": result.summary(),        }    except Exception as e:        return {"error": str(e)}# ---------------------------------------------------------------------------# ML Forecasting Tools# ---------------------------------------------------------------------------def run_ml_forecast(    case_series_json: str,    horizon: int = 14,    dates_json: str = "[]",) -> dict:    """Run ML ensemble forecast (XGBoost + Prophet if available).    Args:        case_series_json: JSON array of daily case counts.        horizon: Number of days to forecast ahead.        dates_json: Optional JSON array of ISO date strings.    Returns:        Dict with ensemble predictions, individual model results, and feature importance.    """    from epiagent.engines.ml_forecaster import forecast_ensemble    try:        case_series = np.array(json.loads(case_series_json), dtype=float)    except (json.JSONDecodeError, TypeError) as e:        return {"error": f"Invalid data: {e}"}    try:        dates_list = json.loads(dates_json) if dates_json != "[]" else None    except json.JSONDecodeError:        dates_list = None    try:        ensemble, individual = forecast_ensemble(            case_series, dates=dates_list, horizon=horizon,        )        return {            "ensemble": ensemble.to_dict(),            "individual_models": [fc.to_dict() for fc in individual],            "model_count": len(individual),        }    except Exception as e:        logger.error("Forecast failed: %s\n%s", e, traceback.format_exc())        return {"error": str(e)}def run_shap_analysis(    case_series_json: str,    top_k: int = 10,) -> dict:    """Run SHAP explainability analysis on epidemic forecast.    Computes SHAP values to explain which features drive the forecast.    Args:        case_series_json: JSON array of daily case counts.        top_k: Number of top features to return.    Returns:        Dict with top drivers, feature importance, and narrative.    """    from epiagent.engines.explainability import explain_xgboost_forecast    try:        case_series = np.array(json.loads(case_series_json), dtype=float)    except (json.JSONDecodeError, TypeError) as e:        return {"error": f"Invalid data: {e}"}    try:        explanation, metrics = explain_xgboost_forecast(            case_series, top_k=top_k,        )        return {            "top_drivers": explanation.top_drivers,            "narrative": explanation.narrative(),            "metrics": metrics,            "global_importance": {                k: round(v, 4)                for k, v in list(explanation.global_importance.items())[:top_k]            },        }    except Exception as e:        logger.error("SHAP analysis failed: %s", e)        return {"error": str(e)}

### Dashboard Generator`epiagent/dashboard/generator.py`

In [ ]:
"""Interactive Plotly Dashboard Generator for EpiAgent.Creates a 7-panel interactive HTML dashboard for epidemic surveillance:    Panel 1: Epidemic Curve (daily cases with 7-day rolling average)    Panel 2: Rt Time Series (with credible intervals + phase shading)    Panel 3: SEIR Model Fit vs Observed    Panel 4: ML Forecast (14-day prediction with uncertainty cone)    Panel 5: SHAP Feature Importance (waterfall chart)    Panel 6: Changepoint Detection overlay    Panel 7: Key Metrics Summary CardsOutput: Self-contained HTML file with Plotly.js embedded."""from __future__ import annotationsimport jsonimport loggingfrom datetime import datetimefrom pathlib import Pathimport numpy as npimport plotly.graph_objects as gofrom plotly.subplots import make_subplotslogger = logging.getLogger(__name__)# ---------------------------------------------------------------------------# Color Palette (modern, accessible)# ---------------------------------------------------------------------------COLORS = {    "primary": "#6366f1",       # Indigo    "secondary": "#8b5cf6",     # Violet    "accent": "#ec4899",        # Pink    "success": "#22c55e",       # Green    "warning": "#f59e0b",       # Amber    "danger": "#ef4444",        # Red    "info": "#06b6d4",          # Cyan    "surface": "#1e1b4b",       # Dark indigo    "text": "#e2e8f0",          # Slate 200    "muted": "#94a3b8",         # Slate 400    "cases": "#6366f1",    "deaths": "#ef4444",    "rt": "#f59e0b",    "forecast": "#22c55e",    "ci_band": "rgba(99, 102, 241, 0.15)",    "rt_band": "rgba(245, 158, 11, 0.15)",    "forecast_band": "rgba(34, 197, 94, 0.15)",}LAYOUT_DEFAULTS = dict(    template="plotly_dark",    paper_bgcolor="#0f0a2a",    plot_bgcolor="#1a1545",    font=dict(family="Inter, system-ui, sans-serif", color=COLORS["text"]),    hovermode="x unified",    hoverlabel=dict(        bgcolor="#2d2a6e",        font_size=12,        bordercolor="#6366f1",    ),)def _create_metric_card_html(    title: str,    value: str,    subtitle: str = "",    color: str = "#6366f1",) -> str:    """Create a metric card HTML element."""    return f"""    <div style="        background: linear-gradient(135deg, {color}22, {color}08);        border: 1px solid {color}44;        border-radius: 16px;        padding: 20px;        text-align: center;        min-width: 160px;    ">        <div style="color: #94a3b8; font-size: 12px; text-transform: uppercase; letter-spacing: 1px;">            {title}        </div>        <div style="color: {color}; font-size: 32px; font-weight: 700; margin: 8px 0;">            {value}        </div>        <div style="color: #64748b; font-size: 11px;">            {subtitle}        </div>    </div>    """# ---------------------------------------------------------------------------# Panel Builders# ---------------------------------------------------------------------------def plot_epidemic_curve(    dates: list[str],    cases: list[int],    deaths: list[int] | None = None,) -> go.Figure:    """Panel 1: Epidemic curve with rolling average."""    fig = go.Figure()    # Daily cases (bar)    fig.add_trace(go.Bar(        x=dates, y=cases,        name="Daily Cases",        marker_color=COLORS["cases"],        opacity=0.5,    ))    # 7-day rolling average    cases_arr = np.array(cases, dtype=float)    if len(cases_arr) >= 7:        rolling_avg = np.convolve(cases_arr, np.ones(7) / 7, mode="valid")        fig.add_trace(go.Scatter(            x=dates[6:], y=rolling_avg,            name="7-Day Average",            line=dict(color=COLORS["accent"], width=3),        ))    # Deaths (if provided)    if deaths:        fig.add_trace(go.Bar(            x=dates, y=deaths,            name="Daily Deaths",            marker_color=COLORS["danger"],            opacity=0.7,            yaxis="y2",        ))    fig.update_layout(        **LAYOUT_DEFAULTS,        title="📊 Epidemic Curve",        xaxis_title="Date",        yaxis_title="Cases",        yaxis2=dict(            title="Deaths",            overlaying="y",            side="right",            showgrid=False,        ),        barmode="overlay",        legend=dict(orientation="h", yanchor="bottom", y=1.08),        margin=dict(t=80),    )    return figdef plot_rt_timeseries(    dates: list[str],    rt_mean: list[float],    rt_lower: list[float],    rt_upper: list[float],    phases: list[str] | None = None,) -> go.Figure:    """Panel 2: Rt time series with CrI and phase shading."""    fig = go.Figure()    # Filter NaN values    valid = [(d, m, l, u) for d, m, l, u in zip(dates, rt_mean, rt_lower, rt_upper)             if m is not None and not (isinstance(m, float) and np.isnan(m))]    if not valid:        fig.add_annotation(text="Insufficient data for Rt estimation",                          xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)        fig.update_layout(**LAYOUT_DEFAULTS, title="📈 Effective Reproduction Number (Rt)")        return fig    v_dates, v_mean, v_lower, v_upper = zip(*valid)    # Credible interval band    fig.add_trace(go.Scatter(        x=list(v_dates) + list(reversed(v_dates)),        y=list(v_upper) + list(reversed(v_lower)),        fill="toself",        fillcolor=COLORS["rt_band"],        line=dict(color="rgba(0,0,0,0)"),        name="95% Credible Interval",        showlegend=True,    ))    # Rt mean line    fig.add_trace(go.Scatter(        x=list(v_dates), y=list(v_mean),        name="Rt (posterior mean)",        line=dict(color=COLORS["rt"], width=3),    ))    # Threshold line at Rt = 1    fig.add_hline(        y=1.0, line_dash="dash", line_color=COLORS["danger"],        annotation_text="Rt = 1 (epidemic threshold)",        annotation_position="bottom right",    )    fig.update_layout(        **LAYOUT_DEFAULTS,        title="📈 Effective Reproduction Number (Rt) — Cori et al. (2013)",        xaxis_title="Date",        yaxis_title="Rt",        yaxis=dict(range=[0, max(5, max(v_upper) * 1.2)]),        legend=dict(orientation="h", yanchor="bottom", y=1.08),        margin=dict(t=80),    )    return figdef plot_seir_fit(    dates: list[str],    observed_cases: list[int],    seir_incidence: list[float],) -> go.Figure:    """Panel 3: SEIR model fit vs observed data."""    fig = go.Figure()    fig.add_trace(go.Scatter(        x=dates[:len(observed_cases)],        y=observed_cases,        name="Observed",        mode="markers",        marker=dict(color=COLORS["cases"], size=4, opacity=0.6),    ))    seir_dates = dates[:len(seir_incidence)]    fig.add_trace(go.Scatter(        x=seir_dates,        y=seir_incidence,        name="SEIR Model",        line=dict(color=COLORS["accent"], width=2.5, dash="dash"),    ))    fig.update_layout(        **LAYOUT_DEFAULTS,        title="🧬 SEIR Model Fit vs Observed Cases",        xaxis_title="Date",        yaxis_title="Daily Incidence",        legend=dict(orientation="h", yanchor="bottom", y=1.08),        margin=dict(t=80),    )    return figdef plot_forecast(    historical_dates: list[str],    historical_cases: list[int],    forecast_dates: list[str],    forecast_predicted: list[float],    forecast_lower: list[float],    forecast_upper: list[float],    model_name: str = "Ensemble",) -> go.Figure:    """Panel 4: ML forecast with uncertainty cone."""    fig = go.Figure()    # Historical    fig.add_trace(go.Scatter(        x=historical_dates[-60:],        y=historical_cases[-60:],        name="Historical Cases",        line=dict(color=COLORS["cases"], width=2),    ))    # Forecast uncertainty band    fig.add_trace(go.Scatter(        x=forecast_dates + list(reversed(forecast_dates)),        y=forecast_upper + list(reversed(forecast_lower)),        fill="toself",        fillcolor=COLORS["forecast_band"],        line=dict(color="rgba(0,0,0,0)"),        name="95% Prediction Interval",    ))    # Forecast line    fig.add_trace(go.Scatter(        x=forecast_dates,        y=forecast_predicted,        name=f"{model_name} Forecast",        line=dict(color=COLORS["forecast"], width=3, dash="dot"),    ))    # Divider    if historical_dates:        fig.add_vline(            x=historical_dates[-1],            line_dash="dash",            line_color=COLORS["muted"],            annotation_text="Forecast →",        )    fig.update_layout(        **LAYOUT_DEFAULTS,        title=f"🔮 14-Day Forecast ({model_name})",        xaxis_title="Date",        yaxis_title="Predicted Cases",        legend=dict(orientation="h", yanchor="bottom", y=1.08),        margin=dict(t=80),    )    return figdef plot_shap_importance(    feature_names: list[str],    importance_values: list[float],) -> go.Figure:    """Panel 5: SHAP feature importance waterfall chart."""    # Sort by importance    pairs = sorted(zip(feature_names, importance_values), key=lambda x: x[1])    names, values = zip(*pairs) if pairs else ([], [])    # Translate names    translations = {        "lag_1": "Yesterday's Cases",        "lag_7": "Cases 1 Week Ago",        "lag_14": "Cases 2 Weeks Ago",        "rolling_7d_mean": "7-Day Average",        "rolling_7d_std": "7-Day Variability",        "rolling_14d_mean": "14-Day Average",        "day_change": "Day-over-Day Change",        "week_change_ratio": "Week Growth Ratio",        "day_of_week": "Day of Week",        "is_weekend": "Weekend Effect",        "month": "Month/Season",    }    display_names = [translations.get(n, n) for n in names]    fig = go.Figure()    fig.add_trace(go.Bar(        x=list(values),        y=display_names,        orientation="h",        marker=dict(            color=list(values),            colorscale=[[0, COLORS["info"]], [1, COLORS["accent"]]],        ),    ))    fig.update_layout(        **LAYOUT_DEFAULTS,        title="🎯 Forecast Driver Analysis (SHAP Feature Importance)",        xaxis_title="Mean |SHAP Value|",        yaxis_title="",        height=400,    )    return figdef plot_changepoints(    dates: list[str],    cases: list[int],    changepoint_indices: list[int],    confidence_scores: list[float] | None = None,) -> go.Figure:    """Panel 6: Case series with changepoint annotations."""    fig = go.Figure()    fig.add_trace(go.Scatter(        x=dates, y=cases,        name="Cases",        line=dict(color=COLORS["cases"], width=2),        fill="tozeroy",        fillcolor="rgba(99, 102, 241, 0.08)",    ))    # Changepoint markers    for i, cp_idx in enumerate(changepoint_indices):        if cp_idx < len(dates):            score = confidence_scores[i] if confidence_scores and i < len(confidence_scores) else 0.5            fig.add_vline(                x=dates[cp_idx],                line_dash="dash",                line_color=COLORS["warning"],                line_width=2,                annotation_text=f"CP (conf: {score:.0%})",                annotation_font_color=COLORS["warning"],            )    fig.update_layout(        **LAYOUT_DEFAULTS,        title="⚡ Changepoint Detection (BOCPD)",        xaxis_title="Date",        yaxis_title="Daily Cases",    )    return fig# ---------------------------------------------------------------------------# Full Dashboard Assembly# ---------------------------------------------------------------------------def generate_dashboard(    surveillance_data: dict,    rt_results: dict | None = None,    seir_results: dict | None = None,    forecast_results: dict | None = None,    shap_results: dict | None = None,    changepoint_results: dict | None = None,    epi_metrics: dict | None = None,    security_report: dict | None = None,    output_path: str = "epiagent_dashboard.html",) -> str:    """Generate complete interactive HTML dashboard.    Args:        surveillance_data: Dict with 'records' list.        rt_results: Rt estimation results.        seir_results: SEIR model results.        forecast_results: ML forecast results.        shap_results: SHAP analysis results.        changepoint_results: Changepoint detection results.        epi_metrics: Epidemiological metrics dict.        security_report: Security audit results.        output_path: Path for the HTML output file.    Returns:        Path to the generated HTML file.    """    records = surveillance_data.get("records", [])    if not records:        logger.warning("No records provided for dashboard")        return ""    dates = [r["date"] for r in records]    cases = [r["new_cases"] for r in records]    deaths = [r.get("new_deaths", 0) for r in records]    panels_html = []    # Panel 1: Epidemic curve    fig1 = plot_epidemic_curve(dates, cases, deaths)    panels_html.append(fig1.to_html(full_html=False, include_plotlyjs=False))    # Panel 2: Rt time series    if rt_results and "rt_last_30d" in rt_results:        rt_data = rt_results["rt_last_30d"]        n_rt = len(rt_data.get("rt_mean", []))        rt_dates = dates[-n_rt:] if n_rt <= len(dates) else dates        fig2 = plot_rt_timeseries(            rt_dates,            rt_data.get("rt_mean", []),            rt_data.get("rt_lower", []),            rt_data.get("rt_upper", []),        )        panels_html.append(fig2.to_html(full_html=False, include_plotlyjs=False))    # Panel 3: SEIR fit    if seir_results and "daily_incidence" in seir_results:        fig3 = plot_seir_fit(dates, cases, seir_results["daily_incidence"])        panels_html.append(fig3.to_html(full_html=False, include_plotlyjs=False))    # Panel 4: Forecast    if forecast_results:        ensemble = forecast_results.get("ensemble", {})        if ensemble:            fig4 = plot_forecast(                dates, cases,                ensemble.get("dates", []),                ensemble.get("predicted", []),                ensemble.get("lower_bound", []),                ensemble.get("upper_bound", []),                ensemble.get("model_name", "Ensemble"),            )            panels_html.append(fig4.to_html(full_html=False, include_plotlyjs=False))    # Panel 5: SHAP    if shap_results and "top_drivers" in shap_results:        drivers = shap_results["top_drivers"]        if drivers:            fig5 = plot_shap_importance(                [d["feature"] for d in drivers],                [d["importance"] for d in drivers],            )            panels_html.append(fig5.to_html(full_html=False, include_plotlyjs=False))    # Panel 6: Changepoints    if changepoint_results and "changepoints" in changepoint_results:        fig6 = plot_changepoints(            dates, cases,            changepoint_results.get("changepoints", []),            changepoint_results.get("confidence_scores", []),        )        panels_html.append(fig6.to_html(full_html=False, include_plotlyjs=False))    # Assemble metric cards    cards = []    if rt_results:        rt_val = rt_results.get("current_rt", "N/A")        phase = rt_results.get("current_phase", "unknown")        rt_color = COLORS["danger"] if phase == "growing" else (            COLORS["success"] if phase == "declining" else COLORS["warning"]        )        cards.append(_create_metric_card_html(            "Current Rt", f"{rt_val:.2f}" if isinstance(rt_val, (int, float)) else str(rt_val),            f"Phase: {phase}", rt_color,        ))    if epi_metrics:        if "cfr" in epi_metrics:            cfr_val = epi_metrics["cfr"]["value"]            cards.append(_create_metric_card_html(                "Case Fatality Rate",                f"{cfr_val:.2%}" if isinstance(cfr_val, (int, float)) else str(cfr_val),                "Wilson score CI", COLORS["danger"],            ))        if "incidence_rate_per_100k" in epi_metrics:            inc = epi_metrics["incidence_rate_per_100k"]["value"]            cards.append(_create_metric_card_html(                "Incidence Rate",                f"{inc:.1f}" if isinstance(inc, (int, float)) else str(inc),                "per 100,000", COLORS["info"],            ))        if "doubling_time_days" in epi_metrics:            dt = epi_metrics["doubling_time_days"]["value"]            cards.append(_create_metric_card_html(                "Doubling Time",                f"{dt:.1f}d" if isinstance(dt, (int, float)) and not np.isnan(dt) else "N/A",                "log-linear regression", COLORS["warning"],            ))    total_cases = sum(cases)    total_deaths = sum(deaths)    cards.append(_create_metric_card_html(        "Total Cases", f"{total_cases:,}", "", COLORS["primary"],    ))    cards.append(_create_metric_card_html(        "Total Deaths", f"{total_deaths:,}", "", COLORS["danger"],    ))    # Security badge    if security_report:        pii = security_report.get("pii_detected", False)        badge_color = COLORS["danger"] if pii else COLORS["success"]        badge_text = "PII DETECTED" if pii else "CLEAN"        cards.append(_create_metric_card_html(            "HIPAA Status", badge_text,            security_report.get("data_hash", "")[:16] + "...",            badge_color,        ))    cards_html = "\n".join(cards)    # Build full HTML    now = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")    pathogen = records[0].get("pathogen", "Unknown") if records else "Unknown"    region = records[0].get("region", "Unknown") if records else "Unknown"    html = f"""<!DOCTYPE html><html lang="en"><head>    <meta charset="UTF-8">    <meta name="viewport" content="width=device-width, initial-scale=1.0">    <title>EpiAgent Dashboard — {pathogen.title()} Surveillance</title>    <script src="https://cdn.plot.ly/plotly-2.35.0.min.js"></script>    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap" rel="stylesheet">    <style>        * {{ box-sizing: border-box; margin: 0; padding: 0; }}        body {{            font-family: 'Inter', system-ui, sans-serif;            background: linear-gradient(135deg, #0f0a2a 0%, #1a1545 50%, #0f172a 100%);            color: #e2e8f0;            min-height: 100vh;        }}        .header {{            background: linear-gradient(135deg, #6366f1 0%, #8b5cf6 100%);            padding: 32px 40px;            display: flex;            justify-content: space-between;            align-items: center;        }}        .header h1 {{            font-size: 28px;            font-weight: 700;            letter-spacing: -0.5px;        }}        .header .subtitle {{            font-size: 14px;            opacity: 0.85;            margin-top: 4px;        }}        .header .badge {{            background: rgba(255,255,255,0.2);            padding: 8px 16px;            border-radius: 20px;            font-size: 13px;        }}        .metrics-bar {{            display: flex;            gap: 16px;            padding: 24px 40px;            overflow-x: auto;            flex-wrap: wrap;        }}        .panels {{            padding: 0 40px 40px;            display: grid;            grid-template-columns: 1fr 1fr;            gap: 24px;        }}        .panel {{            background: rgba(30, 27, 75, 0.6);            border: 1px solid rgba(99, 102, 241, 0.15);            border-radius: 16px;            padding: 16px;            backdrop-filter: blur(10px);        }}        .panel.full-width {{            grid-column: 1 / -1;        }}        .footer {{            text-align: center;            padding: 24px;            color: #64748b;            font-size: 12px;            border-top: 1px solid rgba(99, 102, 241, 0.1);        }}        @media (max-width: 768px) {{            .panels {{ grid-template-columns: 1fr; padding: 0 16px 16px; }}            .metrics-bar {{ padding: 16px; }}            .header {{ padding: 20px 16px; }}        }}    </style></head><body>    <div class="header">        <div>            <h1>🦠 EpiAgent Dashboard</h1>            <div class="subtitle">{pathogen.title()} Surveillance — {region} | Generated: {now}</div>        </div>        <div class="badge">Multi-Agent AI System</div>    </div>    <div class="metrics-bar">        {cards_html}    </div>    <div class="panels">        {"".join(f'<div class="panel{" full-width" if i == 0 else ""}">{p}</div>' for i, p in enumerate(panels_html))}    </div>    <div class="footer">        EpiAgent v1.0 | Google ADK 2.3.0 | Deterministic Engines + LLM Orchestration<br>        Methods: SEIR (RK45) · Cori et al. 2013 (Bayesian Rt) · BOCPD · XGBoost · SHAP<br>        HIPAA Safe Harbor Compliant | Data Hash: {security_report.get("data_hash", "N/A")[:32] if security_report else "N/A"}...    </div></body></html>"""    # Write to file    output = Path(output_path)    output.write_text(html, encoding="utf-8")    logger.info("Dashboard generated: %s", output.resolve())    return str(output.resolve())

## 🤖 Multi-Agent System (ADK 2.3.0)6 specialized LlmAgents orchestrated by a SequentialAgent pipeline.

In [ ]:
"""EpiAgent Multi-Agent Orchestration System.Root agent definition for the Google ADK 2.3.0 framework.This file defines the complete multi-agent graph:    ┌──────────────────────────────────────────────────────────────┐    │                    ORCHESTRATOR (root_agent)                 │    │                  SequentialAgent Pipeline                    │    │                                                              │    │  Step 1: data_agent ──→ Fetch surveillance data              │    │  Step 2: security_agent ──→ PII scan + strip                 │    │  Step 3: validator_agent ──→ Data quality checks             │    │  Step 4: analysis_agent ──→ SEIR, Rt, epi metrics           │    │  Step 5: ml_agent ──→ XGBoost forecast + SHAP               │    │  Step 6: sitrep_agent ──→ Generate executive report          │    └──────────────────────────────────────────────────────────────┘Each agent is an LlmAgent with specialized FunctionTools.The pipeline is a SequentialAgent that passes state via session.ADK Pattern: agent.py must export `root_agent` at module level."""from __future__ import annotationsimport loggingfrom google.adk.agents import (    Agent,    SequentialAgent,)from google.adk.tools import FunctionToolfrom epiagent.agents.tools import (    fetch_synthetic_data,    fetch_cdc_data,    validate_data,    run_security_audit,    run_seir_model,    estimate_rt,    compute_epi_metrics,    detect_changepoints,    run_ml_forecast,    run_shap_analysis,)logger = logging.getLogger(__name__)# ---------------------------------------------------------------------------# Model Configuration# ---------------------------------------------------------------------------MODEL = "gemini-2.0-flash"# ---------------------------------------------------------------------------# Agent 1: Data Retrieval Agent# ---------------------------------------------------------------------------data_agent = Agent(    name="data_retrieval_agent",    model=MODEL,    description="Retrieves epidemic surveillance data from available sources.",    instruction="""You are the Data Retrieval Agent in an epidemic surveillance system.Your job is to fetch surveillance data when requested. You have two data sources:1. **Synthetic data** (fetch_synthetic_data) — For testing/demos. Generates    realistic SEIR-based epidemic data for influenza, COVID-19, or measles.2. **CDC FluView** (fetch_cdc_data) — Real influenza surveillance data from    the CDC via the CMU Delphi Epidata API.When you receive a request:1. Determine the appropriate data source based on the pathogen and context.2. Call the appropriate tool with the requested parameters.3. Store the result by setting state["surveillance_data"] = result.4. Summarize the data you fetched: record count, date range, peak cases.Default behavior: If no specific source is requested, use synthetic COVID-19 data with 180 days, as it demonstrates the system's capabilities well.IMPORTANT: You are a data fetcher only. Do NOT analyze or interpret the data. That is the job of downstream agents.""",    tools=[        FunctionTool(fetch_synthetic_data),        FunctionTool(fetch_cdc_data),    ],)# ---------------------------------------------------------------------------# Agent 2: Security Guardrail Agent# ---------------------------------------------------------------------------security_agent = Agent(    name="security_guardrail_agent",    model=MODEL,    description="Scans data for PII and enforces HIPAA compliance.",    instruction="""You are the Security Guardrail Agent implementing HIPAA Safe Harbor compliance.Your job is to scan all surveillance data for Protected Health Information (PII) before it enters the analysis pipeline.When you receive surveillance data:1. Call run_security_audit with the records as JSON.2. Review the results for any PII detections.3. If PII is detected:   - Report EXACTLY what types were found (email, SSN, phone, etc.)   - Use the cleaned_records from the audit result   - Update state with the cleaned data4. If no PII detected:   - Confirm the data is clean   - Record the data hash for the audit trail5. Store results: state["security_report"] = audit results   state["surveillance_data"]["records"] = cleaned records (if PII was found)CRITICAL: You must NEVER pass data containing PII to downstream agents.If PII is found and cannot be stripped, HALT the pipeline and report.The 18 HIPAA identifier types you scan for include:Names, addresses, dates, phone/fax, email, SSN, medical record numbers,health plan IDs, account numbers, license numbers, VIN, device IDs,URLs, IP addresses, biometric data, photos, and other unique identifiers.""",    tools=[        FunctionTool(run_security_audit),    ],)# ---------------------------------------------------------------------------# Agent 3: Data Validator Agent# ---------------------------------------------------------------------------validator_agent = Agent(    name="data_validator_agent",    model=MODEL,    description="Validates epidemiological data quality with 8-point checks.",    instruction="""You are the Data Validation Agent — the epidemiological "firewall."Your job is to validate surveillance data quality before analysis.When you receive data:1. Call validate_data with the surveillance records as JSON.2. Review the quality report:   - Quality score (0.0 to 1.0): acceptable if >= 0.7   - Errors: critical issues (negative cases, missing fields, zero population)   - Warnings: potential issues (deaths > cases, temporal spikes, non-monotonic cumulative)3. Decision logic:   - Score >= 0.7: PASS — proceed to analysis   - Score 0.5-0.7: CONDITIONAL PASS — note issues but proceed with caveats   - Score < 0.5: FAIL — data quality too low for reliable analysis4. Store: state["quality_report"] = validation resultsThe 8 validation checks are:1. Negative case counts (error)2. Negative death counts (error)  3. Zero population (error)4. Missing required fields (error)5. Deaths exceeding cases (warning)6. Temporal spikes >10x (warning)7. Non-monotonic cumulative counts (warning)8. Missing data completeness (warning)For CONDITIONAL PASS, list the specific caveats that downstream agents should consider in their analysis.""",    tools=[        FunctionTool(validate_data),    ],)# ---------------------------------------------------------------------------# Agent 4: Epidemiological Analysis Agent# ---------------------------------------------------------------------------analysis_agent = Agent(    name="epi_analysis_agent",    model=MODEL,    description="Runs deterministic epidemiological models: SEIR, Rt estimation, metrics.",    instruction="""You are the Epidemiological Analysis Agent — the mathematical core.Your job is to run deterministic epidemiological models and compute key metrics.You have 4 analysis tools:1. **run_seir_model** — SEIR compartmental model simulation2. **estimate_rt** — Bayesian Rt estimation (Cori et al. 2013)3. **compute_epi_metrics** — CFR, incidence rate, attack rate, doubling time4. **detect_changepoints** — BOCPD outbreak signal detectionStandard analysis workflow:1. Extract case series from state["surveillance_data"]["records"]2. Run estimate_rt with the daily case counts and correct pathogen3. Run compute_epi_metrics with totals from the data4. Run detect_changepoints to identify regime changes5. Optionally run run_seir_model to compare observed vs theoretical dynamics6. Store all results in state["epi_analysis"]When presenting Rt results, classify the epidemic phase:- Rt > 1 with lower CI > 1: "Growing — epidemic is expanding"- Rt < 1 with upper CI < 1: "Declining — epidemic is contracting"- CI spans 1: "Stable — uncertain, near reproduction threshold"IMPORTANT: You compute, you do NOT hallucinate. All numbers come from tools.When you report a metric, include the confidence interval and method.""",    tools=[        FunctionTool(run_seir_model),        FunctionTool(estimate_rt),        FunctionTool(compute_epi_metrics),        FunctionTool(detect_changepoints),    ],)# ---------------------------------------------------------------------------# Agent 5: ML Forecasting Agent# ---------------------------------------------------------------------------ml_agent = Agent(    name="ml_forecasting_agent",    model=MODEL,    description="Runs ML forecasting ensemble and SHAP explainability analysis.",    instruction="""You are the ML Forecasting Agent — the predictive intelligence core.Your job is to generate epidemic forecasts and explain what's driving them.You have 2 tools:1. **run_ml_forecast** — XGBoost + Prophet ensemble forecast2. **run_shap_analysis** — SHAP explainability for the forecastWorkflow:1. Extract the case series from state["surveillance_data"]["records"]2. Convert to JSON array of daily new_cases values3. Call run_ml_forecast with horizon=14 (2-week forecast)4. Call run_shap_analysis to explain the forecast drivers5. Store results in state["ml_forecast"]When interpreting SHAP results:- Translate ML feature names to epidemiological language:  - "lag_1" → "yesterday's case count"  - "rolling_7d_mean" → "weekly average trend"  - "week_change_ratio" → "week-over-week growth"  - "is_weekend" → "weekend reporting effect"- Highlight the top 3-5 drivers and explain their epidemiological significance- Note if weekend effects are important (suggests reporting artifacts)Report the forecast uncertainty: include prediction intervals and RMSE.""",    tools=[        FunctionTool(run_ml_forecast),        FunctionTool(run_shap_analysis),    ],)# ---------------------------------------------------------------------------# Agent 6: SitRep Generator Agent# ---------------------------------------------------------------------------sitrep_agent = Agent(    name="sitrep_generator_agent",    model=MODEL,    description="Generates executive situation reports from analysis results.",    instruction="""You are the SitRep Generator Agent — the communication layer.Your job is to synthesize all analysis results into an executive Situation Report(SitRep) suitable for public health decision-makers.Inputs (from state):- state["surveillance_data"] — Raw data summary- state["security_report"] — PII audit results- state["quality_report"] — Data quality assessment- state["epi_analysis"] — SEIR, Rt, metrics, changepoints- state["ml_forecast"] — Forecasts and SHAP explanationsSitRep Structure:1. **ALERT LEVEL** — Assign based on Rt and growth rate:   - 🟢 GREEN: Rt < 0.8, declining epidemic   - 🟡 YELLOW: 0.8 ≤ Rt < 1.0, slow decline   - 🟠 ORANGE: 1.0 ≤ Rt < 1.5, growing epidemic   - 🔴 RED: Rt ≥ 1.5 or rapid exponential growth2. **EXECUTIVE SUMMARY** — 2-3 sentence overview for non-technical readers3. **KEY METRICS TABLE**:   - Current Rt (with 95% CrI)   - Case Fatality Rate (with 95% CI)   - Incidence Rate per 100,000   - Doubling Time (if growing)   - 14-day Forecast Trend4. **EPIDEMIC DYNAMICS** — Phase classification, SEIR comparison, changepoints5. **FORECAST** — 14-day predictions with uncertainty bounds6. **EXPLAINABILITY** — Top SHAP-derived forecast drivers7. **DATA QUALITY** — Quality score, any caveats from validation8. **RECOMMENDATIONS** — 3-5 actionable public health recommendations9. **METHODOLOGY** — Brief description of methods for reproducibilityFormat the SitRep in clean Markdown with tables, bullet points, and clear headers.Use emoji for alert levels. Include timestamps and data provenance.CRITICAL RULES:- NEVER invent numbers. Every metric must come from the analysis tools.- Always include confidence intervals alongside point estimates.- Flag any data quality issues that affect interpretation.- Include the data hash from the security audit for reproducibility.""",    tools=[],  # SitRep agent interprets existing results, no new tools needed)# ---------------------------------------------------------------------------# Root Agent: Sequential Pipeline Orchestrator# ---------------------------------------------------------------------------root_agent = SequentialAgent(    name="epiagent_orchestrator",    description=(        "EpiAgent: Secure Multi-Agent Epidemic Surveillance & Predictive "        "Modeling System. Orchestrates a 6-agent sequential pipeline from "        "data retrieval through analysis to executive situation reports."    ),    sub_agents=[        data_agent,        security_agent,        validator_agent,        analysis_agent,        ml_agent,        sitrep_agent,    ],)

## 🚀 Full Pipeline ExecutionRunning the complete 6-step pipeline:1. Data Retrieval (synthetic COVID-19)2. Security Audit (HIPAA)3. Data Quality Validation4. Epidemiological Analysis (SEIR, Rt, Metrics, Changepoints)5. ML Forecasting (XGBoost + SHAP)6. Dashboard Generation

In [ ]:
"""EpiAgent Full Pipeline Demo — No API Key Required.Runs the complete EpiAgent analysis pipeline using direct function calls(bypassing the LLM agents). This demonstrates all computational enginesand generates the interactive dashboard.This script is perfect for:    1. Kaggle competition submission (runs without API keys)    2. Testing all engines end-to-end    3. Generating the dashboard for screenshots/video    4. Validating the full pipeline before ADK agent wiring"""import jsonimport loggingimport sysimport timefrom pathlib import Pathfrom datetime import datetimeimport numpy as np# Add project root to pathproject_root = Path(__file__).parent# sys.path already set in Kagglefrom epiagent.agents.tools import (    fetch_synthetic_data,    validate_data,    run_security_audit,    run_seir_model,    estimate_rt,    compute_epi_metrics,    detect_changepoints,    run_ml_forecast,    run_shap_analysis,)from epiagent.dashboard.generator import generate_dashboardlogging.basicConfig(    level=logging.INFO,    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",    datefmt="%H:%M:%S",)logger = logging.getLogger("epiagent.demo")# Fix Windows console encodingimport iosys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding='utf-8', errors='replace')def print_banner(text: str, char: str = "="):    """Print a styled banner."""    width = 70    border = char * width    padding = (width - len(text) - 2) // 2    print(f"\n{border}")    print(f"{char}{' ' * padding}{text}{' ' * (width - padding - len(text) - 2)}{char}")    print(f"{border}")def print_metric(name: str, value: str, detail: str = ""):    """Print a metric line."""    print(f"  {'•'} {name:.<35} {value}")    if detail:        print(f"    {'└─'} {detail}")def main():    start_time = time.time()    print_banner("EpiAgent — Full Pipeline Demo")    print(f"  Timestamp: {datetime.utcnow().isoformat()} UTC")    print(f"  Python: {sys.version.split()[0]}")    print()    # =======================================================================    # STEP 1: Data Retrieval    # =======================================================================    print_banner("STEP 1: Data Retrieval", "─")    print("  Generating synthetic COVID-19 epidemic data...")    data = fetch_synthetic_data(        pathogen="covid-19",        duration_days=180,        region="demo_region",        noise_level=0.1,        seed=42,    )    records = data["records"]    print_metric("Records generated", str(data["record_count"]))    print_metric("Date range", data["date_range"])    print_metric("Total cases", f"{data['total_cases']:,}")    print_metric("Total deaths", f"{data['total_deaths']:,}")    print_metric("Peak day", f"{data['peak_day']} ({data['peak_cases']:,} cases)")    # =======================================================================    # STEP 2: Security Audit    # =======================================================================    print_banner("STEP 2: Security Audit (HIPAA)", "─")    print("  Scanning for PII across 18 HIPAA identifier types...")    security_result = run_security_audit(records)    pii_status = "🔴 DETECTED" if security_result["pii_detected"] else "🟢 CLEAN"    print_metric("PII Status", pii_status)    print_metric("Data Hash", security_result["data_hash"][:32] + "...")    print_metric("Schema Valid", str(security_result["schema_valid"]))    if security_result["pii_detected"]:        print_metric("PII Types Found", ", ".join(security_result["pii_types_found"]))        print_metric("Items Redacted", str(security_result["items_redacted"]))        records = security_result["cleaned_records"]    # =======================================================================    # STEP 3: Data Validation    # =======================================================================    print_banner("STEP 3: Data Quality Validation", "─")    print("  Running 8-point epidemiological plausibility checks...")    quality_result = validate_data(records)    score = quality_result["quality_score"]    score_bar = "█" * int(score * 20) + "░" * (20 - int(score * 20))    score_emoji = "🟢" if score >= 0.7 else ("🟡" if score >= 0.5 else "🔴")    print_metric("Quality Score", f"{score:.1%} {score_emoji}")    print(f"    [{score_bar}]")    print_metric("Valid Records", f"{quality_result['valid_records']}/{quality_result['total_records']}")    print_metric("Errors", str(quality_result["error_count"]))    print_metric("Warnings", str(quality_result["warning_count"]))    if quality_result["issues"]:        print("\n  Issues found:")        for issue in quality_result["issues"][:5]:            icon = "❌" if issue["severity"] == "error" else "⚠️"            print(f"    {icon} [{issue['severity']}] {issue['message']}")    # =======================================================================    # STEP 4: Epidemiological Analysis    # =======================================================================    print_banner("STEP 4: Epidemiological Analysis", "─")    # Extract case series    case_series = [r["new_cases"] for r in records]    dates = [r["date"] for r in records]    case_json = json.dumps(case_series)    # 4a: SEIR Model    print("\n  ▸ Running SEIR compartmental model...")    seir_result = run_seir_model(        population=1_000_000, R0=2.5,        latent_period=5.2, infectious_period=2.9,        initial_infected=10, t_max=180,    )    summary = seir_result["summary"]    print_metric("SEIR R0", f"{summary['R0']:.2f}")    print_metric("Peak Day", str(seir_result["peak_day"]))    print_metric("Peak Cases", f"{seir_result['peak_cases']:,.0f}")    print_metric("Attack Rate", f"{seir_result['attack_rate']:.1%}")    # 4b: Rt Estimation    print("\n  ▸ Estimating Rt (Cori et al. 2013)...")    rt_result = estimate_rt(case_json, pathogen="covid-19", window=7)    if "error" not in rt_result:        print_metric("Current Rt", f"{rt_result['current_rt']:.3f}")        print_metric("Epidemic Phase", rt_result["current_phase"].upper())        rt_summary = rt_result.get("summary", {})        if "rt_range" in rt_summary:            print_metric("Rt Range", f"{rt_summary['rt_range'][0]:.2f} – {rt_summary['rt_range'][1]:.2f}")    else:        print(f"  ⚠️ Rt estimation: {rt_result['error']}")    # 4c: Epi Metrics    print("\n  ▸ Computing epidemiological metrics...")    total_cases = sum(case_series)    total_deaths = sum(r["new_deaths"] for r in records)    metrics_result = compute_epi_metrics(        total_cases=total_cases,        total_deaths=total_deaths,        population=1_000_000,        case_series_json=case_json,    )    if "cfr" in metrics_result:        cfr = metrics_result["cfr"]        print_metric("CFR", f"{cfr['value']:.4f}",                     f"95% CI: [{cfr['ci'][0]:.4f}, {cfr['ci'][1]:.4f}] ({cfr['method']})")    if "incidence_rate_per_100k" in metrics_result:        inc = metrics_result["incidence_rate_per_100k"]        print_metric("Incidence Rate", f"{inc['value']:.1f} per 100k",                     f"95% CI: [{inc['ci'][0]:.1f}, {inc['ci'][1]:.1f}]")    if "doubling_time_days" in metrics_result:        dt = metrics_result["doubling_time_days"]        dt_val = f"{dt['value']:.1f} days" if not np.isnan(dt["value"]) else "N/A (declining)"        print_metric("Doubling Time", dt_val)    # 4d: Changepoint Detection    print("\n  ▸ Detecting changepoints (BOCPD)...")    cp_result = detect_changepoints(case_json, hazard_lambda=100.0)    if "error" not in cp_result:        print_metric("Changepoints Found", str(cp_result["n_changepoints"]))        for i, (cp, score) in enumerate(zip(            cp_result["changepoints"][:5],            cp_result["confidence_scores"][:5],        )):            cp_date = dates[cp] if cp < len(dates) else f"day {cp}"            print(f"    CP {i+1}: Day {cp} ({cp_date}) — confidence: {score:.0%}")    else:        print(f"  ⚠️ Changepoint detection: {cp_result['error']}")    # =======================================================================    # STEP 5: ML Forecasting + SHAP    # =======================================================================    print_banner("STEP 5: ML Forecasting & Explainability", "─")    # 5a: Forecast    print("\n  ▸ Running XGBoost+Prophet ensemble forecast...")    forecast_result = run_ml_forecast(case_json, horizon=14, dates_json=json.dumps(dates))    if "error" not in forecast_result:        ensemble = forecast_result["ensemble"]        print_metric("Forecast Model", ensemble["model_name"])        print_metric("Forecast RMSE", f"{ensemble['rmse']:.2f}" if ensemble.get("rmse") else "N/A")        print_metric("14-Day Forecast Range",                     f"{min(ensemble['predicted']):,.0f} – {max(ensemble['predicted']):,.0f}")        print_metric("Models in Ensemble", str(forecast_result["model_count"]))    else:        print(f"  ⚠️ Forecast: {forecast_result['error']}")        forecast_result = None    # 5b: SHAP    print("\n  ▸ Running SHAP explainability analysis...")    shap_result = run_shap_analysis(case_json, top_k=10)    if "error" not in shap_result:        print(f"\n  {shap_result.get('narrative', 'No narrative')}")    else:        print(f"  ⚠️ SHAP: {shap_result['error']}")        shap_result = None    # =======================================================================    # STEP 6: Dashboard Generation    # =======================================================================    print_banner("STEP 6: Dashboard Generation", "─")    print("  Generating interactive Plotly dashboard...")    dashboard_path = generate_dashboard(        surveillance_data=data,        rt_results=rt_result if "error" not in rt_result else None,        seir_results=seir_result,        forecast_results=forecast_result,        shap_results=shap_result,        changepoint_results=cp_result if "error" not in cp_result else None,        epi_metrics=metrics_result,        security_report=security_result,        output_path=str(project_root / "epiagent_dashboard.html"),    )    print_metric("Dashboard Path", dashboard_path)    print_metric("File Size", f"{Path(dashboard_path).stat().st_size / 1024:.0f} KB")    # =======================================================================    # Summary    # =======================================================================    elapsed = time.time() - start_time    print_banner("PIPELINE COMPLETE")    print(f"  ⏱️  Total execution time: {elapsed:.1f} seconds")    print(f"  📊 Dashboard: {dashboard_path}")    print(f"  🔒 HIPAA: {'CLEAN' if not security_result['pii_detected'] else 'REDACTED'}")    print(f"  ✅ Quality: {quality_result['quality_score']:.0%}")    print(f"  📈 Current Rt: {rt_result.get('current_rt', 'N/A')}")    print()    print("  Open the dashboard in your browser to explore interactive visualizations!")    print()    return {        "data": data,        "security": security_result,        "quality": quality_result,        "seir": seir_result,        "rt": rt_result,        "metrics": metrics_result,        "changepoints": cp_result,        "forecast": forecast_result,        "shap": shap_result,        "dashboard_path": dashboard_path,    }if __name__ == "__main__":    main()

## 📊 Multi-Pathogen Scenario ComparisonDemonstrating generalizability across three pathogen profiles:- **COVID-19**: R₀=2.5, serial interval 4.7 days, CFR≈1.5%- **Influenza**: R₀=1.3, serial interval 2.6 days, CFR≈0.1%- **Measles**: R₀=12.0, serial interval 11.5 days, CFR≈0.2%

In [ ]:
"""Multi-Pathogen Scenario Testing.Runs the full EpiAgent pipeline across three pathogen profilesto demonstrate the system's generalizability:    1. COVID-19 (R0=2.5, SI=4.7d)    2. Influenza (R0=1.3, SI=2.6d)    3. Measles (R0=12.0, SI=11.5d)Generates per-pathogen dashboards and a comparative summary."""import jsonimport sysimport timeimport iofrom pathlib import Pathimport numpy as npproject_root = Path(__file__).parent# sys.path already set in Kagglesys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding='utf-8', errors='replace')from epiagent.agents.tools import (    fetch_synthetic_data,    validate_data,    run_security_audit,    run_seir_model,    estimate_rt,    compute_epi_metrics,    detect_changepoints,    run_ml_forecast,    run_shap_analysis,)from epiagent.dashboard.generator import generate_dashboardSCENARIOS = [    {        "pathogen": "covid-19",        "display_name": "COVID-19",        "R0": 2.5,        "latent_period": 5.2,        "infectious_period": 2.9,        "duration": 180,    },    {        "pathogen": "influenza",        "display_name": "Influenza",        "R0": 1.3,        "latent_period": 2.0,        "infectious_period": 3.0,        "duration": 120,    },    {        "pathogen": "measles",        "display_name": "Measles",        "R0": 12.0,        "latent_period": 10.0,        "infectious_period": 8.0,        "duration": 180,    },]def run_scenario(scenario: dict) -> dict:    """Run full pipeline for one pathogen scenario."""    name = scenario["display_name"]    print(f"\n{'='*60}")    print(f"  SCENARIO: {name} (R0={scenario['R0']})")    print(f"{'='*60}")    t0 = time.time()    # Step 1: Data    data = fetch_synthetic_data(        pathogen=scenario["pathogen"],        duration_days=scenario["duration"],        region=f"{name.lower()}_region",    )    records = data["records"]    case_series = [r["new_cases"] for r in records]    dates = [r["date"] for r in records]    case_json = json.dumps(case_series)    print(f"  Data: {data['record_count']} records, {data['total_cases']:,} cases")    # Step 2: Security    security = run_security_audit(records)    print(f"  Security: {'CLEAN' if not security['pii_detected'] else 'PII FOUND'}")    # Step 3: Validation    quality = validate_data(records)    print(f"  Quality: {quality['quality_score']:.0%}")    # Step 4: Analysis    seir = run_seir_model(        population=1_000_000,        R0=scenario["R0"],        latent_period=scenario["latent_period"],        infectious_period=scenario["infectious_period"],        initial_infected=10,        t_max=scenario["duration"],    )    rt = estimate_rt(case_json, pathogen=scenario["pathogen"], window=7)    total_cases = sum(case_series)    total_deaths = sum(r["new_deaths"] for r in records)    metrics = compute_epi_metrics(        total_cases=total_cases,        total_deaths=total_deaths,        population=1_000_000,        case_series_json=case_json,    )    cp = detect_changepoints(case_json)    print(f"  Rt: {rt.get('current_rt', 'N/A'):.3f}" if isinstance(rt.get('current_rt'), float) else f"  Rt: {rt.get('current_rt', 'N/A')}")    print(f"  Phase: {rt.get('current_phase', 'N/A')}")    # Step 5: Forecast    forecast = run_ml_forecast(case_json, horizon=14, dates_json=json.dumps(dates))    shap = run_shap_analysis(case_json, top_k=5)    if "error" not in forecast:        ensemble = forecast["ensemble"]        print(f"  Forecast RMSE: {ensemble.get('rmse', 'N/A'):.2f}" if ensemble.get('rmse') else "  Forecast: done")    # Step 6: Dashboard    dashboard_path = generate_dashboard(        surveillance_data=data,        rt_results=rt if "error" not in rt else None,        seir_results=seir,        forecast_results=forecast if "error" not in forecast else None,        shap_results=shap if "error" not in shap else None,        changepoint_results=cp if "error" not in cp else None,        epi_metrics=metrics,        security_report=security,        output_path=str(project_root / f"dashboard_{scenario['pathogen'].replace('-','_')}.html"),    )    elapsed = time.time() - t0    print(f"  Dashboard: {dashboard_path}")    print(f"  Time: {elapsed:.1f}s")    return {        "pathogen": name,        "R0": scenario["R0"],        "total_cases": total_cases,        "total_deaths": total_deaths,        "cfr": metrics.get("cfr", {}).get("value", float("nan")),        "current_rt": rt.get("current_rt", float("nan")),        "phase": rt.get("current_phase", "unknown"),        "quality_score": quality["quality_score"],        "forecast_rmse": forecast.get("ensemble", {}).get("rmse") if "error" not in forecast else None,        "dashboard": dashboard_path,        "elapsed": elapsed,    }def main():    print("=" * 60)    print("  EpiAgent Multi-Pathogen Scenario Testing")    print("=" * 60)    results = []    for scenario in SCENARIOS:        result = run_scenario(scenario)        results.append(result)    # Comparative summary    print("\n" + "=" * 60)    print("  COMPARATIVE SUMMARY")    print("=" * 60)    print(f"\n  {'Pathogen':<12} {'R0':>5} {'Cases':>12} {'Deaths':>8} {'CFR':>8} {'Rt':>6} {'Phase':<10} {'Time':>5}")    print("  " + "-" * 75)    for r in results:        cfr_str = f"{r['cfr']:.4f}" if not np.isnan(r['cfr']) else "N/A"        rt_str = f"{r['current_rt']:.3f}" if isinstance(r['current_rt'], float) and not np.isnan(r['current_rt']) else "N/A"        print(f"  {r['pathogen']:<12} {r['R0']:>5.1f} {r['total_cases']:>12,} {r['total_deaths']:>8,} {cfr_str:>8} {rt_str:>6} {r['phase']:<10} {r['elapsed']:>4.1f}s")    total_time = sum(r["elapsed"] for r in results)    print(f"\n  Total time: {total_time:.1f}s for {len(results)} scenarios")    print(f"  Dashboards generated in project root directory")    return resultsif __name__ == "__main__":    main()

## 🏁 Conclusion### What EpiAgent Demonstrates1. **Responsible AI for Healthcare**: HIPAA-compliant PII scanning, deterministic computation, and full audit trails ensure AI systems can be trusted in public health contexts.2. **Multi-Agent Orchestration**: Google ADK 2.3.0's SequentialAgent pattern provides a clean, maintainable pipeline architecture where each agent has a single responsibility.3. **Rigorous Statistical Methods**: Using Wilson score intervals (not Wald), exact Poisson CIs, and the Cori et al. (2013) Bayesian Rt method — the actual WHO/CDC standard.4. **Explainable AI**: SHAP TreeExplainer provides post-hoc explanations for every forecast, critical for public health decision-making.5. **Production-Ready Engineering**: 55 passing unit tests, 3.1-second pipeline execution, and modular architecture ready for deployment.### References1. Cori A, Ferguson NM, Fraser C, Cauchemez S. (2013) "A New Framework and Software to Estimate Time-Varying Reproduction Numbers During Epidemics." *Am J Epidemiol*, 178(9):1505-1512.2. Adams RP, MacKay DJC. (2007) "Bayesian Online Changepoint Detection." arXiv:0710.3742.3. Wilson EB. (1927) "Probable Inference, the Law of Succession, and Statistical Inference." *JASA*, 22(158):209-212.4. Lundberg SM, Lee SI. (2017) "A Unified Approach to Interpreting Model Predictions." *NeurIPS*.5. Kermack WO, McKendrick AG. (1927) "A Contribution to the Mathematical Theory of Epidemics." *Proc Royal Soc A*, 115(772):700-721.6. Chen T, Guestrin C. (2016) "XGBoost: A Scalable Tree Boosting System." *KDD*.---*Built with ❤️ for public health.*